In [1]:
# Inspect official LINZ New Zealand LiDAR 1 m DEM
# STAC collection

import requests
from urllib.parse import urljoin
from pathlib import Path


# 1. Detect project root automatically

CURRENT_FOLDER = Path.cwd()
PROJECT_ROOT = None

for folder in [
    CURRENT_FOLDER,
    *CURRENT_FOLDER.parents
]:
    candidate = (
        folder
        / "data"
    )

    if candidate.exists():
        PROJECT_ROOT = folder
        break


if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the project root containing the data folder."
    )


TERRAIN_DIR = (
    PROJECT_ROOT
    / "data"
    / "terrain"
)

TERRAIN_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# Official LINZ national 1 m bare-earth DEM collection

DEM_COLLECTION_URL = (
    "https://nz-elevation.s3-ap-southeast-2.amazonaws.com/"
    "new-zealand/new-zealand/dem_1m/2193/collection.json"
)


# 2. Connect to official LINZ STAC collection

response = requests.get(
    DEM_COLLECTION_URL,
    timeout=60
)

response.raise_for_status()

dem_collection = response.json()


print(
    "LINZ DEM Collection"
)

print(
    "Title:",
    dem_collection.get(
        "title"
    )
)

print(
    "Description:",
    dem_collection.get(
        "description"
    )
)

print(
    "License:",
    dem_collection.get(
        "license"
    )
)

print(
    "STAC version:",
    dem_collection.get(
        "stac_version"
    )
)


# 3. Find DEM tile metadata records

item_links = [
    link
    for link in dem_collection[
        "links"
    ]
    if link.get(
        "rel"
    ) == "item"
]


print(
    "\nNumber of national DEM tile records:",
    len(item_links)
)


print(
    "\nFirst 10 tile metadata files:"
)

for link in item_links[
    :10
]:
    print(
        link[
            "href"
        ]
    )


# 4. Inspect one tile metadata record
# No TIFF download

sample_item_url = urljoin(
    DEM_COLLECTION_URL,
    item_links[
        0
    ][
        "href"
    ]
)


sample_response = requests.get(
    sample_item_url,
    timeout=60
)

sample_response.raise_for_status()

sample_item = sample_response.json()


print(
    "\nExample DEM Tile Metadata"
)

print(
    "Tile ID:",
    sample_item.get(
        "id"
    )
)

print(
    "Bounding box:",
    sample_item.get(
        "bbox"
    )
)

print(
    "Available assets:"
)

for asset_name, asset in (
    sample_item
    .get(
        "assets",
        {}
    )
    .items()
):

    print(
        asset_name,
        "->",
        asset.get(
            "href"
        )
    )


print(
    "\nNo DEM raster has been downloaded."
)

LINZ DEM Collection
Title: New Zealand LiDAR 1m DEM
Description: Digital Elevation Model within the New Zealand region captured in 2008-2026.
License: CC-BY-4.0
STAC version: 1.0.0

Number of national DEM tile records: 424

First 10 tile metadata files:
./AS21.json
./AS22.json
./AT24.json
./AT25.json
./AU25.json
./AU26.json
./AU27.json
./AU28.json
./AU29.json
./AV25.json

Example DEM Tile Metadata
Tile ID: AS21
Bounding box: [171.8256487, -34.3559317, 172.090076, -34.0291036]
Available assets:
visual -> ./AS21.tiff

No DEM raster has been downloaded.


In [2]:
# Build LINZ DEM tile index and identify tiles
# containing CAS crash locations

from pathlib import Path

import requests
import pandas as pd
import numpy as np
import geopandas as gpd

from shapely.geometry import box
from urllib.parse import urljoin
from concurrent.futures import (
    ThreadPoolExecutor,
    as_completed
)


# 1. Detect project root automatically

CURRENT_FOLDER = Path.cwd()
PROJECT_ROOT = None


for folder in [
    CURRENT_FOLDER,
    *CURRENT_FOLDER.parents
]:

    candidate = (
        folder
        / "data"
        / "CAS_Data_public.csv"
    )

    if candidate.exists():

        PROJECT_ROOT = folder
        break


if PROJECT_ROOT is None:

    raise FileNotFoundError(
        "Could not find the project root containing "
        "data/CAS_Data_public.csv."
    )


TERRAIN_DIR = (
    PROJECT_ROOT
    / "data"
    / "terrain"
)


TERRAIN_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print(
    "Project root:",
    PROJECT_ROOT
)


# 2. Function to read one STAC tile record

def read_dem_item(link):

    item_url = urljoin(
        DEM_COLLECTION_URL,
        link["href"]
    )

    r = requests.get(
        item_url,
        timeout=60
    )

    r.raise_for_status()

    item = r.json()

    bbox = item["bbox"]

    visual_asset = (
        item
        .get("assets", {})
        .get("visual", {})
        .get("href")
    )

    tiff_url = (
        urljoin(
            item_url,
            visual_asset
        )
        if visual_asset
        else None
    )

    return {
        "tile_id":
            item["id"],

        "west":
            bbox[0],

        "south":
            bbox[1],

        "east":
            bbox[2],

        "north":
            bbox[3],

        "item_url":
            item_url,

        "tiff_url":
            tiff_url
    }


# 3. Download metadata for all DEM tiles
# JSON only, not the TIFF rasters

print(
    "\nReading metadata for",
    len(item_links),
    "DEM tiles..."
)


tile_records = []


with ThreadPoolExecutor(
    max_workers=12
) as executor:

    futures = [
        executor.submit(
            read_dem_item,
            link
        )
        for link in item_links
    ]


    for i, future in enumerate(
        as_completed(futures),
        start=1
    ):

        tile_records.append(
            future.result()
        )

        if (
            i % 50 == 0
            or i == len(futures)
        ):

            print(
                f"Metadata read: "
                f"{i}/{len(futures)}"
            )


dem_tile_index = (
    pd.DataFrame(
        tile_records
    )
    .sort_values(
        "tile_id"
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nDEM tile index shape:",
    dem_tile_index.shape
)

print(
    "Missing TIFF URLs:",
    dem_tile_index[
        "tiff_url"
    ]
    .isna()
    .sum()
)


# 4. Convert tile bounding boxes to polygons
# STAC bbox coordinates are longitude/latitude
# EPSG:4326

dem_tiles_gdf = gpd.GeoDataFrame(

    dem_tile_index.copy(),

    geometry=[
        box(
            row.west,
            row.south,
            row.east,
            row.north
        )

        for row in (
            dem_tile_index
            .itertuples()
        )
    ],

    crs="EPSG:4326"
)


print(
    "\nTile-index CRS:",
    dem_tiles_gdf.crs
)


# 5. Load CAS

CAS_PATH = (
    PROJECT_ROOT
    / "data"
    / "CAS_Data_public.csv"
)


print(
    "\nCAS file:"
)

print(
    CAS_PATH
)

print(
    "CAS file exists:",
    CAS_PATH.exists()
)


cas = pd.read_csv(
    CAS_PATH,
    low_memory=False
)


cas[
    "cas_row_id"
] = np.arange(
    len(cas)
)


print(
    "\nCAS rows:",
    len(cas)
)


# 6. Create crash points in NZTM2000

crash_points_2193 = (
    gpd.GeoDataFrame(

        cas[
            [
                "cas_row_id",
                "X",
                "Y",
                "crashSeverity"
            ]
        ].copy(),

        geometry=
            gpd.points_from_xy(
                cas["X"],
                cas["Y"]
            ),

        crs="EPSG:2193"
    )
)


# 7. Transform crash locations to geographic coordinates
# so they match STAC bbox coordinates

crash_points_4326 = (
    crash_points_2193
    .to_crs(
        "EPSG:4326"
    )
)


print(
    "Crash CRS after transformation:",
    crash_points_4326.crs
)


# 8. Spatially match crash points to DEM tile bounding boxes

crash_tile_candidates = (
    gpd.sjoin(

        crash_points_4326[
            [
                "cas_row_id",
                "crashSeverity",
                "geometry"
            ]
        ],

        dem_tiles_gdf[
            [
                "tile_id",
                "tiff_url",
                "geometry"
            ]
        ],

        how="left",

        predicate="intersects"
    )
)


# 9. Coverage statistics

candidate_counts = (
    crash_tile_candidates
    .groupby(
        "cas_row_id"
    )[
        "tile_id"
    ]
    .count()
)


covered_crashes = (
    candidate_counts > 0
).sum()


uncovered_crashes = (
    candidate_counts == 0
).sum()


multiple_candidate_crashes = (
    candidate_counts > 1
).sum()


print(
    "\nDEM Tile Candidate Coverage"
)

print(
    "Total crashes:",
    len(cas)
)

print(
    "Crashes inside at least one DEM bbox:",
    covered_crashes
)

print(
    "Crashes outside all DEM bboxes:",
    uncovered_crashes
)

print(
    "Coverage rate:",
    round(
        covered_crashes
        / len(cas)
        * 100,
        2
    ),
    "%"
)

print(
    "Crashes intersecting multiple tile bboxes:",
    multiple_candidate_crashes
)


# 10. Number of different DEM tiles needed

required_tiles = (
    crash_tile_candidates[
        "tile_id"
    ]
    .dropna()
    .unique()
)


print(
    "\nTotal national DEM tiles:",
    len(dem_tile_index)
)

print(
    "DEM tiles potentially needed:",
    len(required_tiles)
)


# 11. Crash count by candidate DEM tile

tile_crash_counts = (
    crash_tile_candidates
    .dropna(
        subset=[
            "tile_id"
        ]
    )
    .groupby(
        "tile_id"
    )[
        "cas_row_id"
    ]
    .nunique()
    .sort_values(
        ascending=False
    )
)


print(
    "\nTop 20 DEM Tiles by Crash Count"
)

display(
    tile_crash_counts
    .head(20)
    .rename(
        "Crash_Count"
    )
    .reset_index()
)


# 12. Inspect crashes outside all DEM tile bboxes

uncovered_ids = (
    candidate_counts[
        candidate_counts == 0
    ]
    .index
)


uncovered_context_cols = [
    col
    for col in [
        "cas_row_id",
        "X",
        "Y",
        "crashSeverity",
        "region",
        "tlaName",
        "crashLocation1",
        "crashLocation2",
        "urban"
    ]
    if col in cas.columns
]


uncovered_dem_crashes = (
    cas.loc[
        cas[
            "cas_row_id"
        ].isin(
            uncovered_ids
        ),
        uncovered_context_cols
    ]
)


print(
    "\nCrashes Outside All DEM Tile Bounding Boxes"
)

print(
    "Count:",
    len(
        uncovered_dem_crashes
    )
)


display(
    uncovered_dem_crashes
    .head(30)
)


# 13. Save small reusable DEM tile index

DEM_INDEX_PATH = (
    TERRAIN_DIR
    / "LINZ_DEM_1m_tile_index.csv"
)


dem_tile_index.to_csv(
    DEM_INDEX_PATH,
    index=False
)


print(
    "\nSaved DEM tile index:"
)

print(
    DEM_INDEX_PATH
)


print(
    "\nNo DEM TIFF raster has been downloaded."
)


Project root: C:\Users\ASUS\Desktop\Project_Roadcrash

Reading metadata for 424 DEM tiles...
Metadata read: 50/424
Metadata read: 100/424
Metadata read: 150/424
Metadata read: 200/424
Metadata read: 250/424
Metadata read: 300/424
Metadata read: 350/424
Metadata read: 400/424
Metadata read: 424/424

DEM tile index shape: (424, 7)
Missing TIFF URLs: 0

Tile-index CRS: EPSG:4326

CAS file:
C:\Users\ASUS\Desktop\Project_Roadcrash\data\CAS_Data_public.csv
CAS file exists: True

CAS rows: 34866
Crash CRS after transformation: EPSG:4326

DEM Tile Candidate Coverage
Total crashes: 34866
Crashes inside at least one DEM bbox: 34851
Crashes outside all DEM bboxes: 15
Coverage rate: 99.96 %
Crashes intersecting multiple tile bboxes: 1647

Total national DEM tiles: 424
DEM tiles potentially needed: 338

Top 20 DEM Tiles by Crash Count


,tile_id,Crash_Count
0,BA31,3790
1,BA32,3536
2,BB32,3440
3,BX24,1953
4,BD33,1358
5,BQ31,1069
6,BD37,910
7,BK39,905
8,BM34,876
9,BQ32,707



Crashes Outside All DEM Tile Bounding Boxes
Count: 15


,cas_row_id,X,Y,crashSeverity,region,tlaName,crashLocation1,crashLocation2,urban
3719,3719,1.237577e+06,5.017661e+06,Non-Injury Crash,Otago Region,Queenstown-Lakes District,GLENORCHY-QUEENSTOWN ROAD,SHEPHERDS HUT CRK BR,Open
3755,3755,1.235564e+06,5.031091e+06,Serious Crash,Otago Region,Queenstown-Lakes District,GLENORCHY-PARADISE ROAD,PRIORY ROAD,Open
5607,5607,1.237894e+06,5.017414e+06,Serious Crash,Otago Region,Queenstown-Lakes District,GLENORCHY-QUEENSTOWN ROAD,SHEPHERDS HUT CRK BR,Open
6841,6841,1.238262e+06,5.017225e+06,Non-Injury Crash,Otago Region,Queenstown-Lakes District,GLENORCHY-QUEENSTOWN ROAD,SHEPHERDS HUT CRK BR,Open
10078,10078,2.444711e+06,5.076666e+06,Serious Crash,Manawatū-Whanganui Region,Chatham Islands Territory,WAITANGI WHARF OWENGA ROAD,TE MATARAE ROAD,Open
11627,11627,1.236077e+06,5.028898e+06,Non-Injury Crash,Otago Region,Queenstown-Lakes District,GLENORCHY-PARADISE ROAD,NEREUS WAY,Open
18308,18308,2.443069e+06,5.077164e+06,Minor Crash,Manawatū-Whanganui Region,Chatham Islands Territory,WAITANGI WHARF OWENGA ROAD,TE MATARAE ROAD,Open
21608,21608,2.441217e+06,5.091483e+06,Non-Injury Crash,Manawatū-Whanganui Region,Chatham Islands Territory,NORTH ROAD,AIR BASE ROAD,Open
22715,22715,1.238996e+06,5.015392e+06,Non-Injury Crash,Otago Region,Queenstown-Lakes District,GLENORCHY-QUEENSTOWN ROAD,SHEPHERDS HUT CRK BR,Open
24703,24703,1.237991e+06,5.017391e+06,Minor Crash,Otago Region,Queenstown-Lakes District,GLENORCHY-QUEENSTOWN ROAD,SHEPHERDS HUT CRK BR,Open



Saved DEM tile index:
C:\Users\ASUS\Desktop\Project_Roadcrash\data\terrain\LINZ_DEM_1m_tile_index.csv

No DEM TIFF raster has been downloaded.


In [3]:
%pip install rasterio

Note: you may need to restart the kernel to use updated packages.


In [4]:
# Test remote reading from one LINZ DEM COG
# No full TIFF download

import requests
import rasterio
import numpy as np
import pandas as pd


TEST_TILE = "BA31"


# 1. Get remote TIFF URL

test_tile_row = (
    dem_tile_index.loc[
        dem_tile_index[
            "tile_id"
        ] == TEST_TILE
    ]
    .iloc[0]
)


test_tiff_url = (
    test_tile_row[
        "tiff_url"
    ]
)


print(
    "Test DEM Tile"
)

print(
    "Tile:",
    TEST_TILE
)

print(
    "\nRemote TIFF:"
)

print(
    test_tiff_url
)


# 2. Inspect HTTP information
# HEAD request only, no raster download

head = requests.head(
    test_tiff_url,
    allow_redirects=True,
    timeout=60
)


print(
    "\nRemote File Information"
)

print(
    "HTTP status:",
    head.status_code
)

print(
    "Content type:",
    head.headers.get(
        "Content-Type"
    )
)

print(
    "Accept ranges:",
    head.headers.get(
        "Accept-Ranges"
    )
)


content_length = (
    head.headers.get(
        "Content-Length"
    )
)


if content_length is not None:

    size_gb = (
        int(content_length)
        / 1024
        / 1024
        / 1024
    )

    print(
        "Full TIFF size:",
        round(
            size_gb,
            3
        ),
        "GB"
    )

else:

    print(
        "Full TIFF size:",
        "Not reported"
    )


# 3. Choose 10 crashes whose bbox intersects BA31

sample_crash_ids = (
    crash_tile_candidates.loc[
        crash_tile_candidates[
            "tile_id"
        ] == TEST_TILE,
        "cas_row_id"
    ]
    .drop_duplicates()
    .head(10)
    .tolist()
)


sample_points = (
    crash_points_2193.loc[
        crash_points_2193[
            "cas_row_id"
        ].isin(
            sample_crash_ids
        )
    ]
    .copy()
)


print(
    "\nSample crashes:",
    len(sample_points)
)


# 4. Open remote COG
# GDAL reads only required parts of the remote file

with rasterio.Env(

    GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",

    CPL_VSIL_CURL_ALLOWED_EXTENSIONS=(
        ".tif,.tiff"
    ),

    GDAL_HTTP_MULTIRANGE="YES",

    GDAL_HTTP_MERGE_CONSECUTIVE_RANGES="YES"

):

    with rasterio.open(
        test_tiff_url
    ) as src:

        # Raster metadata

        print(
            "\nDEM Raster Metadata"
        )

        print(
            "Driver:",
            src.driver
        )

        print(
            "CRS:",
            src.crs
        )

        print(
            "Width:",
            src.width
        )

        print(
            "Height:",
            src.height
        )

        print(
            "Resolution:",
            src.res
        )

        print(
            "Bounds:",
            src.bounds
        )

        print(
            "Bands:",
            src.count
        )

        print(
            "Data type:",
            src.dtypes
        )

        print(
            "NoData:",
            src.nodata
        )

        print(
            "Block shapes:",
            src.block_shapes
        )


        # Ensure crash points use raster CRS

        sample_points_raster = (
            sample_points
            .to_crs(
                src.crs
            )
        )


        coordinates = [
            (
                geom.x,
                geom.y
            )
            for geom in (
                sample_points_raster.geometry
            )
        ]


        # Read elevation values at crash locations

        sampled_values = list(
            src.sample(
                coordinates,
                indexes=1,
                masked=True
            )
        )


# 5. Build result table

elevations = []


for value in sampled_values:

    pixel = value[0]

    if np.ma.is_masked(
        pixel
    ):

        elevations.append(
            np.nan
        )

    else:

        elevations.append(
            float(pixel)
        )


sample_elevation_results = (
    sample_points_raster[
        [
            "cas_row_id",
            "X",
            "Y",
            "crashSeverity"
        ]
    ]
    .copy()
)


sample_elevation_results[
    "Elevation_m"
] = elevations


print(
    "\nSample Remote Elevation Results"
)

display(
    sample_elevation_results
)


print(
    "\nValid elevation values:",
    sample_elevation_results[
        "Elevation_m"
    ]
    .notna()
    .sum(),
    "/",
    len(
        sample_elevation_results
    )
)


print(
    "\nNo full DEM TIFF was downloaded."
)

Test DEM Tile
Tile: BA31

Remote TIFF:
https://nz-elevation.s3-ap-southeast-2.amazonaws.com/new-zealand/new-zealand/dem_1m/2193/BA31.tiff

Remote File Information
HTTP status: 200
Content type: image/tiff; application=geotiff; profile=cloud-optimized
Accept ranges: bytes
Full TIFF size: 1.178 GB

Sample crashes: 10

DEM Raster Metadata
Driver: GTiff
CRS: EPSG:2193
Width: 24000
Height: 36000
Resolution: (1.0, 1.0)
Bounds: BoundingBox(left=1732000.0, bottom=5910000.0, right=1756000.0, top=5946000.0)
Bands: 1
Data type: ('float32',)
NoData: -9999.0
Block shapes: [(512, 512)]

Sample Remote Elevation Results


,cas_row_id,X,Y,crashSeverity,Elevation_m
27,27,1.750515e+06,5.915770e+06,Serious Crash,17.050749
36,36,1.755516e+06,5.924732e+06,Non-Injury Crash,27.801001
43,43,1.755376e+06,5.932234e+06,Non-Injury Crash,61.767250
56,56,1.741988e+06,5.913162e+06,Non-Injury Crash,138.830994
57,57,1.752898e+06,5.918168e+06,Non-Injury Crash,13.534250
73,73,1.732357e+06,5.943648e+06,Non-Injury Crash,14.315500
74,74,1.744654e+06,5.918031e+06,Non-Injury Crash,31.443249
81,81,1.749539e+06,5.914146e+06,Non-Injury Crash,13.999999
102,102,1.752816e+06,5.918791e+06,Minor Crash,10.218499
107,107,1.743658e+06,5.919391e+06,Non-Injury Crash,12.989250



Valid elevation values: 10 / 10

No full DEM TIFF was downloaded.


In [5]:
# Read exact STAC footprints with retries and reuse completed metadata.

import time
import requests
import pandas as pd
import geopandas as gpd

from shapely.geometry import shape
from urllib.parse import urljoin
from concurrent.futures import ThreadPoolExecutor, as_completed


def read_dem_item_exact(link):
    item_url = urljoin(DEM_COLLECTION_URL, link["href"])

    for attempt in range(5):
        try:
            response = requests.get(item_url, timeout=(15, 60))
            response.raise_for_status()
            item = response.json()

            visual_asset = (
                item.get("assets", {})
                .get("visual", {})
                .get("href")
            )

            return {
                "tile_id": item["id"],
                "item_url": item_url,
                "tiff_url": (
                    urljoin(item_url, visual_asset)
                    if visual_asset else None
                ),
                "geometry": shape(item["geometry"])
            }

        except requests.RequestException:
            if attempt == 4:
                raise
            time.sleep(min(2 ** attempt, 8))


# Retain successful records from an interrupted run.
existing_records = globals().get("exact_records", [])

records_by_url = {
    record["item_url"]: record
    for record in existing_records
    if isinstance(record, dict) and record.get("item_url")
}

expected_urls = {
    urljoin(DEM_COLLECTION_URL, link["href"])
    for link in item_links
}

# Keep only records belonging to the current collection.
records_by_url = {
    url: record
    for url, record in records_by_url.items()
    if url in expected_urls
}

exact_records = list(records_by_url.values())

pending_links = [
    link for link in item_links
    if urljoin(DEM_COLLECTION_URL, link["href"])
    not in records_by_url
]

print(f"DEM tiles: {len(expected_urls)}")
print(f"Completed metadata reused: {len(exact_records)}")
print(f"Metadata still required: {len(pending_links)}")

failed_items = []

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {
        executor.submit(read_dem_item_exact, link): link
        for link in pending_links
    }

    for completed, future in enumerate(as_completed(futures), start=1):
        link = futures[future]

        try:
            record = future.result()
            records_by_url[record["item_url"]] = record

            # Update immediately so an interrupted run can resume.
            exact_records = list(records_by_url.values())

        except Exception as error:
            item_url = urljoin(DEM_COLLECTION_URL, link["href"])
            failed_items.append({
                "item_url": item_url,
                "error": str(error)
            })
            print(f"Failed after retries: {item_url}")

        if completed % 25 == 0 or completed == len(futures):
            print(
                f"Metadata available: "
                f"{len(exact_records)}/{len(expected_urls)}"
            )

if failed_items:
    display(pd.DataFrame(failed_items))
    raise RuntimeError(
        f"{len(failed_items)} metadata requests failed. "
        "Rerun this cell to retry the remaining tiles. "
        "Successful records are retained in this session."
    )

if set(records_by_url) != expected_urls:
    raise ValueError("Tile metadata is incomplete. Do not continue.")

dem_tiles_exact = (
    gpd.GeoDataFrame(
        exact_records,
        geometry="geometry",
        crs="EPSG:4326"
    )
    .sort_values("tile_id")
    .reset_index(drop=True)
)

if dem_tiles_exact["tile_id"].duplicated().any():
    raise ValueError("Duplicate DEM tile IDs detected.")

if (
    dem_tiles_exact.geometry.isna().any()
    or dem_tiles_exact.geometry.is_empty.any()
):
    raise ValueError("Missing or empty DEM footprints detected.")

print("\nExact tile index shape:", dem_tiles_exact.shape)
print("CRS:", dem_tiles_exact.crs)

crash_tile_exact = gpd.sjoin(
    crash_points_4326[
        ["cas_row_id", "crashSeverity", "geometry"]
    ],
    dem_tiles_exact[
        ["tile_id", "tiff_url", "geometry"]
    ],
    how="left",
    predicate="intersects"
)

exact_candidate_counts = (
    crash_tile_exact.groupby("cas_row_id")["tile_id"].count()
)

exact_covered = (exact_candidate_counts > 0).sum()
exact_uncovered = (exact_candidate_counts == 0).sum()
exact_multiple = (exact_candidate_counts > 1).sum()

required_exact_tiles = (
    crash_tile_exact["tile_id"].dropna().unique()
)

print("\nExact STAC Footprint Coverage")
print("Total crashes:", len(cas))
print("Crashes inside at least one footprint:", exact_covered)
print("Crashes outside all footprints:", exact_uncovered)
print(f"Coverage rate: {exact_covered / len(cas):.2%}")
print("Crashes intersecting multiple footprints:", exact_multiple)
print("National DEM tiles:", len(dem_tiles_exact))
print("Exact DEM tiles required:", len(required_exact_tiles))

print("\nNumber of Tile Candidates per Crash")
print(exact_candidate_counts.value_counts().sort_index())

multi_exact_ids = exact_candidate_counts[
    exact_candidate_counts > 1
].index

if len(multi_exact_ids) > 0:
    display(
        crash_tile_exact.loc[
            crash_tile_exact["cas_row_id"].isin(multi_exact_ids),
            ["cas_row_id", "crashSeverity", "tile_id"]
        ]
        .sort_values(["cas_row_id", "tile_id"])
        .head(40)
    )

exact_uncovered_ids = exact_candidate_counts[
    exact_candidate_counts == 0
].index

context_columns = [
    col for col in [
        "cas_row_id", "X", "Y", "crashSeverity", "region",
        "tlaName", "crashLocation1", "crashLocation2"
    ]
    if col in cas.columns
]

exact_uncovered_context = cas.loc[
    cas["cas_row_id"].isin(exact_uncovered_ids),
    context_columns
].copy()

print("\nCrashes Outside Exact DEM Footprints")
print("Count:", len(exact_uncovered_context))
display(exact_uncovered_context)

DEM tiles: 424
Completed metadata reused: 0
Metadata still required: 424
Metadata available: 25/424
Metadata available: 50/424
Metadata available: 75/424
Metadata available: 100/424
Metadata available: 125/424
Metadata available: 150/424
Metadata available: 175/424
Metadata available: 200/424
Metadata available: 225/424
Metadata available: 250/424
Metadata available: 275/424
Metadata available: 300/424
Metadata available: 325/424
Metadata available: 350/424
Metadata available: 375/424
Metadata available: 400/424
Metadata available: 424/424

Exact tile index shape: (424, 4)
CRS: EPSG:4326

Exact STAC Footprint Coverage
Total crashes: 34866
Crashes inside at least one footprint: 34851
Crashes outside all footprints: 15
Coverage rate: 99.96%
Crashes intersecting multiple footprints: 0
National DEM tiles: 424
Exact DEM tiles required: 335

Number of Tile Candidates per Crash
tile_id
0       15
1    34851
Name: count, dtype: int64

Crashes Outside Exact DEM Footprints
Count: 15


,cas_row_id,X,Y,crashSeverity,region,tlaName,crashLocation1,crashLocation2
3719,3719,1.237577e+06,5.017661e+06,Non-Injury Crash,Otago Region,Queenstown-Lakes District,GLENORCHY-QUEENSTOWN ROAD,SHEPHERDS HUT CRK BR
3755,3755,1.235564e+06,5.031091e+06,Serious Crash,Otago Region,Queenstown-Lakes District,GLENORCHY-PARADISE ROAD,PRIORY ROAD
5607,5607,1.237894e+06,5.017414e+06,Serious Crash,Otago Region,Queenstown-Lakes District,GLENORCHY-QUEENSTOWN ROAD,SHEPHERDS HUT CRK BR
6841,6841,1.238262e+06,5.017225e+06,Non-Injury Crash,Otago Region,Queenstown-Lakes District,GLENORCHY-QUEENSTOWN ROAD,SHEPHERDS HUT CRK BR
10078,10078,2.444711e+06,5.076666e+06,Serious Crash,Manawatū-Whanganui Region,Chatham Islands Territory,WAITANGI WHARF OWENGA ROAD,TE MATARAE ROAD
11627,11627,1.236077e+06,5.028898e+06,Non-Injury Crash,Otago Region,Queenstown-Lakes District,GLENORCHY-PARADISE ROAD,NEREUS WAY
18308,18308,2.443069e+06,5.077164e+06,Minor Crash,Manawatū-Whanganui Region,Chatham Islands Territory,WAITANGI WHARF OWENGA ROAD,TE MATARAE ROAD
21608,21608,2.441217e+06,5.091483e+06,Non-Injury Crash,Manawatū-Whanganui Region,Chatham Islands Territory,NORTH ROAD,AIR BASE ROAD
22715,22715,1.238996e+06,5.015392e+06,Non-Injury Crash,Otago Region,Queenstown-Lakes District,GLENORCHY-QUEENSTOWN ROAD,SHEPHERDS HUT CRK BR
24703,24703,1.237991e+06,5.017391e+06,Minor Crash,Otago Region,Queenstown-Lakes District,GLENORCHY-QUEENSTOWN ROAD,SHEPHERDS HUT CRK BR


In [6]:
# Fast block-based elevation extraction test for BA32
# Each raster block is read only once,
# then all crash points inside that block are extracted.

import rasterio
from rasterio.windows import Window

import numpy as np
import pandas as pd
import time


TEST_TILE = "BA32"


# 1. Create crash-to-DEM tile lookup if needed

if "crash_tile_lookup" not in globals():

    crash_tile_lookup = (
        crash_tile_exact[
            [
                "cas_row_id",
                "tile_id",
                "tiff_url"
            ]
        ]
        .dropna(
            subset=[
                "tile_id"
            ]
        )
        .drop_duplicates(
            subset=[
                "cas_row_id"
            ]
        )
        .merge(
            cas[
                [
                    "cas_row_id",
                    "X",
                    "Y",
                    "crashSeverity"
                ]
            ],
            on="cas_row_id",
            how="left",
            validate="one_to_one"
        )
    )


print(
    "Crash-to-DEM tile lookup rows:",
    len(crash_tile_lookup)
)


# 2. Get BA32 crashes

tile_data = (
    crash_tile_lookup.loc[
        crash_tile_lookup[
            "tile_id"
        ].astype(str) == TEST_TILE
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


if tile_data.empty:

    raise ValueError(
        f"No crash records were found for DEM tile {TEST_TILE}."
    )


tiff_url = (
    tile_data[
        "tiff_url"
    ]
    .iloc[0]
)


if pd.isna(
    tiff_url
):

    raise ValueError(
        f"No TIFF URL was found for DEM tile {TEST_TILE}."
    )


print(
    "\nTile:",
    TEST_TILE
)

print(
    "Crash points:",
    len(tile_data)
)


elevations = np.full(
    len(tile_data),
    np.nan,
    dtype=float
)


start_time = time.time()


# 3. Open remote COG

with rasterio.Env(

    GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",

    CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tif,.tiff",

    GDAL_HTTP_MULTIRANGE="YES",

    GDAL_HTTP_MERGE_CONSECUTIVE_RANGES="YES",

    GDAL_CACHEMAX=512

):

    with rasterio.open(
        tiff_url
    ) as src:


        block_height, block_width = (
            src.block_shapes[0]
        )


        print(
            "Raster size:",
            src.width,
            "x",
            src.height
        )

        print(
            "Block size:",
            block_width,
            "x",
            block_height
        )


        # 4. Convert crash X/Y to raster row and column

        rows, cols = rasterio.transform.rowcol(
            src.transform,
            tile_data[
                "X"
            ].to_numpy(),
            tile_data[
                "Y"
            ].to_numpy()
        )


        rows = np.asarray(
            rows,
            dtype=int
        )

        cols = np.asarray(
            cols,
            dtype=int
        )


        inside = (
            (rows >= 0)
            & (rows < src.height)
            & (cols >= 0)
            & (cols < src.width)
        )


        print(
            "Crash points inside raster bounds:",
            inside.sum()
        )

        print(
            "Crash points outside raster bounds:",
            (~inside).sum()
        )


        # 5. Determine raster block containing each crash

        point_index = pd.DataFrame(
            {
                "data_index":
                    np.arange(
                        len(tile_data)
                    ),

                "row":
                    rows,

                "col":
                    cols,

                "inside":
                    inside
            }
        )


        point_index = (
            point_index.loc[
                point_index[
                    "inside"
                ]
            ]
            .copy()
        )


        point_index[
            "block_row"
        ] = (
            point_index[
                "row"
            ]
            // block_height
        )


        point_index[
            "block_col"
        ] = (
            point_index[
                "col"
            ]
            // block_width
        )


        grouped = (
            point_index
            .groupby(
                [
                    "block_row",
                    "block_col"
                ],
                sort=False
            )
        )


        print(
            "\nUnique raster blocks required:",
            grouped.ngroups
        )


        # 6. Read every required block once

        for block_number, (
            (block_row, block_col),
            group
        ) in enumerate(
            grouped,
            start=1
        ):


            row_off = (
                int(block_row)
                * block_height
            )

            col_off = (
                int(block_col)
                * block_width
            )


            window_height = min(
                block_height,
                src.height
                - row_off
            )

            window_width = min(
                block_width,
                src.width
                - col_off
            )


            window = Window(
                col_off=col_off,
                row_off=row_off,
                width=window_width,
                height=window_height
            )


            block = src.read(
                1,
                window=window,
                masked=True
            )


            # Extract every crash inside this block

            for record in (
                group.itertuples()
            ):


                local_row = (
                    record.row
                    - row_off
                )

                local_col = (
                    record.col
                    - col_off
                )


                value = block[
                    local_row,
                    local_col
                ]


                if np.ma.is_masked(
                    value
                ):

                    elevations[
                        record.data_index
                    ] = np.nan

                else:

                    value = float(
                        value
                    )


                    if (
                        src.nodata is not None
                        and np.isclose(
                            value,
                            src.nodata
                        )
                    ):

                        elevations[
                            record.data_index
                        ] = np.nan

                    else:

                        elevations[
                            record.data_index
                        ] = value


            if (
                block_number % 50 == 0
                or block_number
                == grouped.ngroups
            ):

                print(
                    f"Blocks processed: "
                    f"{block_number}/"
                    f"{grouped.ngroups}"
                )


# 7. Results

tile_test_result = (
    tile_data[
        [
            "cas_row_id",
            "X",
            "Y",
            "crashSeverity"
        ]
    ]
    .copy()
)


tile_test_result[
    "Elevation_m"
] = elevations


elapsed = (
    time.time()
    - start_time
)


print(
    "\nBA32 Block Extraction Result"
)


print(
    "Crash points:",
    len(tile_test_result)
)


print(
    "Valid elevations:",
    tile_test_result[
        "Elevation_m"
    ]
    .notna()
    .sum()
)


print(
    "Missing elevations:",
    tile_test_result[
        "Elevation_m"
    ]
    .isna()
    .sum()
)


print(
    "Elapsed seconds:",
    round(
        elapsed,
        2
    )
)


print(
    "Elapsed minutes:",
    round(
        elapsed / 60,
        2
    )
)


print(
    "\nElevation summary:"
)

print(
    tile_test_result[
        "Elevation_m"
    ]
    .describe()
)


print(
    "\nFirst 10 results:"
)

display(
    tile_test_result.head(
        10
    )
)

Crash-to-DEM tile lookup rows: 34851

Tile: BA32
Crash points: 3245
Raster size: 24000 x 36000
Block size: 512 x 512
Crash points inside raster bounds: 3245
Crash points outside raster bounds: 0

Unique raster blocks required: 589
Blocks processed: 50/589
Blocks processed: 100/589
Blocks processed: 150/589
Blocks processed: 200/589
Blocks processed: 250/589
Blocks processed: 300/589
Blocks processed: 350/589
Blocks processed: 400/589
Blocks processed: 450/589
Blocks processed: 500/589
Blocks processed: 550/589
Blocks processed: 589/589

BA32 Block Extraction Result
Crash points: 3245
Valid elevations: 3245
Missing elevations: 0
Elapsed seconds: 124.66
Elapsed minutes: 2.08

Elevation summary:
count    3245.000000
mean       33.844239
std        25.880360
min        -2.621250
25%        12.021749
50%        26.098000
75%        54.651501
max       142.678757
Name: Elevation_m, dtype: float64

First 10 results:


,cas_row_id,X,Y,crashSeverity,Elevation_m
0,0,1.764525e+06,5.912700e+06,Non-Injury Crash,6.689000
1,12,1.763821e+06,5.913705e+06,Non-Injury Crash,22.412249
2,21,1.761496e+06,5.913264e+06,Non-Injury Crash,21.335749
3,29,1.765372e+06,5.916673e+06,Non-Injury Crash,15.405750
4,38,1.768763e+06,5.911738e+06,Minor Crash,8.961000
5,55,1.756476e+06,5.924437e+06,Non-Injury Crash,4.110750
6,59,1.762956e+06,5.915000e+06,Non-Injury Crash,38.306747
7,72,1.759349e+06,5.918705e+06,Minor Crash,3.024500
8,75,1.769779e+06,5.912934e+06,Minor Crash,10.728250
9,83,1.757100e+06,5.919438e+06,Non-Injury Crash,68.622757


In [7]:
# Create the initial BA32 checkpoint only if no checkpoint exists.

ELEVATION_CHECKPOINT_PATH = (
    TERRAIN_DIR / "terrain_elevation_checkpoint.csv"
)

TERRAIN_DIR.mkdir(parents=True, exist_ok=True)

if ELEVATION_CHECKPOINT_PATH.exists():
    print("Existing checkpoint preserved:")
    print(ELEVATION_CHECKPOINT_PATH)
    print("No BA32 results were written.")

else:
    ba32_checkpoint = tile_test_result[
        ["cas_row_id", "Elevation_m"]
    ].copy()

    if ba32_checkpoint.empty:
        raise ValueError("BA32 results are empty. Check Cell 6.")

    if (
        ba32_checkpoint["cas_row_id"].isna().any()
        or ba32_checkpoint["cas_row_id"].duplicated().any()
    ):
        raise ValueError(
            "BA32 results contain missing or duplicate crash IDs."
        )

    ba32_checkpoint["tile_id"] = "BA32"

    ba32_checkpoint = ba32_checkpoint[
        ["cas_row_id", "tile_id", "Elevation_m"]
    ]

    # Exclusive creation also prevents overwriting an existing file.
    ba32_checkpoint.to_csv(
        ELEVATION_CHECKPOINT_PATH,
        index=False,
        mode="x"
    )

    print("Created initial BA32 checkpoint:")
    print(ELEVATION_CHECKPOINT_PATH)
    print(f"Rows saved: {len(ba32_checkpoint):,}")
    print(
        "Valid elevations:",
        ba32_checkpoint["Elevation_m"].notna().sum()
    )
    print(
        "Missing elevations:",
        ba32_checkpoint["Elevation_m"].isna().sum()
    )

Existing checkpoint preserved:
C:\Users\ASUS\Desktop\Project_Roadcrash\data\terrain\terrain_elevation_checkpoint.csv
No BA32 results were written.


In [8]:
# Diagnose BA32 raster-block usage
# No full elevation extraction yet

import rasterio
import numpy as np
import pandas as pd


TEST_TILE = "BA32"


tile_data = (
    crash_tile_lookup.loc[
        crash_tile_lookup[
            "tile_id"
        ] == TEST_TILE
    ]
    .copy()
)


tiff_url = (
    tile_data[
        "tiff_url"
    ]
    .iloc[0]
)


print(
    "Tile:",
    TEST_TILE
)

print(
    "Crash points:",
    len(tile_data)
)


with rasterio.Env(
    GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
    CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tif,.tiff",
    GDAL_HTTP_MULTIRANGE="YES",
    GDAL_HTTP_MERGE_CONSECUTIVE_RANGES="YES"
):

    with rasterio.open(
        tiff_url
    ) as src:

        block_height, block_width = (
            src.block_shapes[0]
        )


        print(
            "Raster size:",
            src.width,
            "x",
            src.height
        )

        print(
            "Block size:",
            block_width,
            "x",
            block_height
        )


        # Convert crash coordinates to raster rows and columns

        rows, cols = rasterio.transform.rowcol(
            src.transform,
            tile_data[
                "X"
            ].to_numpy(),
            tile_data[
                "Y"
            ].to_numpy()
        )


        rows = np.asarray(
            rows
        )

        cols = np.asarray(
            cols
        )


        # Identify which raster block contains each crash point

        block_rows = (
            rows
            // block_height
        )

        block_cols = (
            cols
            // block_width
        )


        block_table = pd.DataFrame(
            {
                "row":
                    rows,

                "col":
                    cols,

                "block_row":
                    block_rows,

                "block_col":
                    block_cols
            }
        )


        unique_blocks = (
            block_table[
                [
                    "block_row",
                    "block_col"
                ]
            ]
            .drop_duplicates()
        )


        print(
            "\nUnique raster blocks needed:",
            len(unique_blocks)
        )


        crashes_per_block = (
            block_table
            .groupby(
                [
                    "block_row",
                    "block_col"
                ]
            )
            .size()
        )


        print(
            "\nCrashes per required block:"
        )

        print(
            crashes_per_block.describe()
        )


        print(
            "\nMaximum crashes in one block:",
            crashes_per_block.max()
        )


print(
    "\nNo full DEM raster was downloaded."
)

Tile: BA32
Crash points: 3245
Raster size: 24000 x 36000
Block size: 512 x 512

Unique raster blocks needed: 589

Crashes per required block:
count    589.000000
mean       5.509338
std        7.252163
min        1.000000
25%        2.000000
50%        3.000000
75%        6.000000
max       77.000000
dtype: float64

Maximum crashes in one block: 77

No full DEM raster was downloaded.


In [9]:
# Faster parallel terrain elevation extraction
# 1. 512x512 raster-block reads instead of point-by-point reads
# 2. Multiple DEM tiles processed simultaneously
# 3. Existing checkpoint automatically resumed
# 4. Saves after every completed tile
# 5. Failed tiles do not stop the whole job

import rasterio
from rasterio.windows import Window

import numpy as np
import pandas as pd

from concurrent.futures import (
    ThreadPoolExecutor,
    as_completed
)

import time


# 1. Settings

MAX_WORKERS = 6


ELEVATION_CHECKPOINT_PATH = (
    TERRAIN_DIR
    / "terrain_elevation_checkpoint.csv"
)

ELEVATION_FINAL_PATH = (
    TERRAIN_DIR
    / "CAS_LINZ_elevation_raw.csv"
)

FAILED_TILES_PATH = (
    TERRAIN_DIR
    / "terrain_failed_tiles.csv"
)


print(
    "Parallel workers:",
    MAX_WORKERS
)


# 2. Create crash-to-exact-DEM-tile lookup

crash_tile_lookup = (
    crash_tile_exact[
        [
            "cas_row_id",
            "tile_id",
            "tiff_url"
        ]
    ]
    .dropna(
        subset=[
            "tile_id"
        ]
    )
    .drop_duplicates(
        subset=[
            "cas_row_id"
        ]
    )
    .merge(
        cas[
            [
                "cas_row_id",
                "X",
                "Y",
                "crashSeverity"
            ]
        ],
        on="cas_row_id",
        how="left",
        validate="one_to_one"
    )
)


print(
    "\nTerrain Extraction Setup"
)

print(
    "Covered crashes:",
    len(crash_tile_lookup)
)

print(
    "Unique DEM tiles:",
    crash_tile_lookup[
        "tile_id"
    ].nunique()
)

print(
    "Duplicate crash IDs:",
    crash_tile_lookup[
        "cas_row_id"
    ]
    .duplicated()
    .sum()
)


# 3. Load checkpoint

if ELEVATION_CHECKPOINT_PATH.exists():

    elevation_results = pd.read_csv(
        ELEVATION_CHECKPOINT_PATH
    )

    elevation_results = (
        elevation_results
        .drop_duplicates(
            subset=[
                "cas_row_id"
            ],
            keep="last"
        )
        .reset_index(
            drop=True
        )
    )


    completed_tiles = set(
        elevation_results[
            "tile_id"
        ]
        .dropna()
        .astype(str)
        .unique()
    )


    print(
        "\nCheckpoint found."
    )

    print(
        "Completed tiles:",
        len(completed_tiles)
    )

    print(
        "Crashes already saved:",
        len(elevation_results)
    )

    print(
        "Valid elevations already saved:",
        elevation_results[
            "Elevation_m"
        ]
        .notna()
        .sum()
    )

else:

    elevation_results = pd.DataFrame(
        columns=[
            "cas_row_id",
            "tile_id",
            "Elevation_m"
        ]
    )

    completed_tiles = set()

    print(
        "\nNo checkpoint found."
    )


# 4. Remaining tile order
# Largest crash-count tiles first

tile_sizes = (
    crash_tile_lookup
    .groupby(
        "tile_id"
    )
    .size()
    .sort_values(
        ascending=False
    )
)


all_tiles = (
    tile_sizes
    .index
    .astype(str)
    .tolist()
)


remaining_tiles = [
    tile
    for tile in all_tiles
    if tile not in completed_tiles
]


print(
    "\nTotal required tiles:",
    len(all_tiles)
)

print(
    "Already completed:",
    len(completed_tiles)
)

print(
    "Tiles remaining:",
    len(remaining_tiles)
)


print(
    "\nFirst 10 remaining tiles:"
)

print(
    remaining_tiles[
        :10
    ]
)


# 5. Remote COG configuration

RASTER_ENV_OPTIONS = {

    "GDAL_DISABLE_READDIR_ON_OPEN":
        "EMPTY_DIR",

    "CPL_VSIL_CURL_ALLOWED_EXTENSIONS":
        ".tif,.tiff",

    "GDAL_HTTP_MULTIRANGE":
        "YES",

    "GDAL_HTTP_MERGE_CONSECUTIVE_RANGES":
        "YES",

    "GDAL_HTTP_MAX_RETRY":
        "3",

    "GDAL_HTTP_RETRY_DELAY":
        "1",

    "GDAL_CACHEMAX":
        512
}


# 6. Fast block extraction for one tile

def extract_one_tile(
    tile_id
):

    tile_start = time.time()


    tile_data = (
        crash_tile_lookup.loc[
            crash_tile_lookup[
                "tile_id"
            ].astype(str) == str(tile_id)
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    if len(tile_data) == 0:

        raise ValueError(
            f"{tile_id}: no crash records found"
        )


    tiff_url = (
        tile_data[
            "tiff_url"
        ]
        .iloc[0]
    )


    elevations = np.full(
        len(tile_data),
        np.nan,
        dtype=float
    )


    blocks_read = 0


    # Each thread opens its own raster dataset

    with rasterio.Env(
        **RASTER_ENV_OPTIONS
    ):

        with rasterio.open(
            tiff_url
        ) as src:


            if str(
                src.crs
            ) != "EPSG:2193":

                raise ValueError(
                    f"{tile_id}: "
                    f"unexpected CRS {src.crs}"
                )


            block_height, block_width = (
                src.block_shapes[0]
            )


            # Crash coordinates to raster rows and columns

            rows, cols = (
                rasterio.transform.rowcol(
                    src.transform,
                    tile_data[
                        "X"
                    ].to_numpy(),
                    tile_data[
                        "Y"
                    ].to_numpy()
                )
            )


            rows = np.asarray(
                rows,
                dtype=int
            )

            cols = np.asarray(
                cols,
                dtype=int
            )


            inside = (
                (rows >= 0)
                & (rows < src.height)
                & (cols >= 0)
                & (cols < src.width)
            )


            points = pd.DataFrame(
                {
                    "data_index":
                        np.arange(
                            len(tile_data)
                        ),

                    "row":
                        rows,

                    "col":
                        cols,

                    "inside":
                        inside
                }
            )


            points = (
                points.loc[
                    points[
                        "inside"
                    ]
                ]
                .copy()
            )


            points[
                "block_row"
            ] = (
                points[
                    "row"
                ]
                // block_height
            )


            points[
                "block_col"
            ] = (
                points[
                    "col"
                ]
                // block_width
            )


            grouped = (
                points
                .groupby(
                    [
                        "block_row",
                        "block_col"
                    ],
                    sort=False
                )
            )


            # Read each required raster block once

            for (
                block_row,
                block_col
            ), group in grouped:


                row_off = (
                    int(
                        block_row
                    )
                    * block_height
                )


                col_off = (
                    int(
                        block_col
                    )
                    * block_width
                )


                height = min(
                    block_height,
                    src.height
                    - row_off
                )


                width = min(
                    block_width,
                    src.width
                    - col_off
                )


                window = Window(
                    col_off=col_off,
                    row_off=row_off,
                    width=width,
                    height=height
                )


                block = src.read(
                    1,
                    window=window,
                    masked=True
                )


                blocks_read += 1


                # Vectorised extraction of all crashes
                # within this raster block

                local_rows = (
                    group[
                        "row"
                    ]
                    .to_numpy()
                    - row_off
                )


                local_cols = (
                    group[
                        "col"
                    ]
                    .to_numpy()
                    - col_off
                )


                data_indices = (
                    group[
                        "data_index"
                    ]
                    .to_numpy()
                )


                values = block[
                    local_rows,
                    local_cols
                ]


                # Convert masked array safely

                if np.ma.isMaskedArray(
                    values
                ):

                    valid_mask = (
                        ~np.ma.getmaskarray(
                            values
                        )
                    )

                    raw_values = (
                        np.asarray(
                            values.data,
                            dtype=float
                        )
                    )

                else:

                    valid_mask = np.ones(
                        len(values),
                        dtype=bool
                    )

                    raw_values = np.asarray(
                        values,
                        dtype=float
                    )


                valid_mask &= np.isfinite(
                    raw_values
                )


                if src.nodata is not None:

                    valid_mask &= (
                        ~np.isclose(
                            raw_values,
                            float(
                                src.nodata
                            )
                        )
                    )


                valid_indices = (
                    data_indices[
                        valid_mask
                    ]
                )


                elevations[
                    valid_indices
                ] = (
                    raw_values[
                        valid_mask
                    ]
                )


    # Result

    result = (
        tile_data[
            [
                "cas_row_id",
                "tile_id"
            ]
        ]
        .copy()
    )


    result[
        "Elevation_m"
    ] = elevations


    elapsed = (
        time.time()
        - tile_start
    )


    return {

        "tile_id":
            str(tile_id),

        "result":
            result,

        "crashes":
            len(result),

        "valid":
            int(
                result[
                    "Elevation_m"
                ]
                .notna()
                .sum()
            ),

        "missing":
            int(
                result[
                    "Elevation_m"
                ]
                .isna()
                .sum()
            ),

        "blocks":
            blocks_read,

        "seconds":
            elapsed
    }


# 7. Parallel extraction

failed_tiles = []

session_start = time.time()


print(
    "\nStarting Parallel DEM Extraction"
)

print(
    f"Processing up to {MAX_WORKERS} "
    "DEM tiles simultaneously."
)


with ThreadPoolExecutor(
    max_workers=MAX_WORKERS
) as executor:


    future_to_tile = {

        executor.submit(
            extract_one_tile,
            tile
        ):
        tile

        for tile in remaining_tiles
    }


    completed_this_session = 0


    for future in as_completed(
        future_to_tile
    ):


        tile_id = (
            future_to_tile[
                future
            ]
        )


        try:

            output = (
                future.result()
            )


            tile_result = (
                output[
                    "result"
                ]
            )


            # Add result

            elevation_results = (
                pd.concat(
                    [
                        elevation_results,
                        tile_result
                    ],
                    ignore_index=True
                )
                .drop_duplicates(
                    subset=[
                        "cas_row_id"
                    ],
                    keep="last"
                )
                .reset_index(
                    drop=True
                )
            )


            completed_tiles.add(
                tile_id
            )


            completed_this_session += 1


            # Save after every finished tile

            elevation_results.to_csv(
                ELEVATION_CHECKPOINT_PATH,
                index=False
            )


            print(
                f"\nCompleted: {tile_id}"
            )

            print(
                "Crashes:",
                output[
                    "crashes"
                ]
            )

            print(
                "Blocks:",
                output[
                    "blocks"
                ]
            )

            print(
                "Valid:",
                output[
                    "valid"
                ]
            )

            print(
                "Missing:",
                output[
                    "missing"
                ]
            )

            print(
                "Time:",
                round(
                    output[
                        "seconds"
                    ],
                    1
                ),
                "sec"
            )

            print(
                "Progress:",
                len(
                    completed_tiles
                ),
                "/",
                len(
                    all_tiles
                ),
                "tiles"
            )


            # Progress summary every 10 finished tiles

            if (
                completed_this_session
                % 10
                == 0
            ):

                elapsed_minutes = (
                    time.time()
                    - session_start
                ) / 60


                print(
                    "\nParallel Progress Summary"
                )

                print(
                    "Completed this session:",
                    completed_this_session
                )

                print(
                    "Completed total:",
                    len(
                        completed_tiles
                    ),
                    "/",
                    len(
                        all_tiles
                    )
                )

                print(
                    "Crashes saved:",
                    len(
                        elevation_results
                    )
                )

                print(
                    "Valid elevations:",
                    elevation_results[
                        "Elevation_m"
                    ]
                    .notna()
                    .sum()
                )

                print(
                    "Elapsed:",
                    round(
                        elapsed_minutes,
                        1
                    ),
                    "minutes"
                )

                print(
                    "Failed tiles:",
                    len(
                        failed_tiles
                    )
                )


        except KeyboardInterrupt:

            print(
                "\nManual stop requested."
            )

            print(
                "Finished tiles are already saved."
            )

            raise


        except Exception as e:

            failed_tiles.append(
                {
                    "tile_id":
                        tile_id,

                    "error":
                        repr(e)
                }
            )


            print(
                f"\nFailed: {tile_id}"
            )

            print(
                repr(e)
            )


            pd.DataFrame(
                failed_tiles
            ).to_csv(
                FAILED_TILES_PATH,
                index=False
            )


# 8. Build final all-crash dataset

terrain_elevation = (
    cas[
        [
            "cas_row_id",
            "X",
            "Y",
            "crashSeverity"
        ]
    ]
    .merge(
        elevation_results[
            [
                "cas_row_id",
                "tile_id",
                "Elevation_m"
            ]
        ],
        on="cas_row_id",
        how="left",
        validate="one_to_one"
    )
)


# 9. Final validation

valid_total = (
    terrain_elevation[
        "Elevation_m"
    ]
    .notna()
    .sum()
)


missing_total = (
    terrain_elevation[
        "Elevation_m"
    ]
    .isna()
    .sum()
)


print(
    "\nFinal Elevation Extraction"
)


print(
    "Total crashes:",
    len(
        terrain_elevation
    )
)


print(
    "Valid elevations:",
    valid_total
)


print(
    "Missing elevations:",
    missing_total
)


print(
    "Elevation coverage:",
    round(
        valid_total
        / len(
            terrain_elevation
        )
        * 100,
        2
    ),
    "%"
)


print(
    "Duplicate crash IDs:",
    terrain_elevation[
        "cas_row_id"
    ]
    .duplicated()
    .sum()
)


# 10. Elevation statistics

print(
    "\nElevation Summary (metres)"
)


print(
    terrain_elevation[
        "Elevation_m"
    ]
    .describe()
)


print(
    "\nElevation < -20 m:",
    (
        terrain_elevation[
            "Elevation_m"
        ] < -20
    ).sum()
)


print(
    "Elevation > 3000 m:",
    (
        terrain_elevation[
            "Elevation_m"
        ] > 3000
    ).sum()
)


# 11. Missing cases

missing_elevation = (
    terrain_elevation.loc[
        terrain_elevation[
            "Elevation_m"
        ].isna()
    ]
)


print(
    "\nMissing Elevation Cases"
)

print(
    "Count:",
    len(
        missing_elevation
    )
)


display(
    missing_elevation
)


# 12. Failed tiles

failed_tiles_df = pd.DataFrame(
    failed_tiles
)


print(
    "\nFailed DEM Tiles"
)

print(
    "Count:",
    len(
        failed_tiles_df
    )
)


if len(
    failed_tiles_df
) > 0:

    display(
        failed_tiles_df
    )


# 13. Save final elevation file

terrain_elevation.to_csv(
    ELEVATION_FINAL_PATH,
    index=False
)


print(
    "\nSaved final elevation file:"
)

print(
    ELEVATION_FINAL_PATH
)


print(
    "\nCheckpoint retained:"
)

print(
    ELEVATION_CHECKPOINT_PATH
)

Parallel workers: 6

Terrain Extraction Setup
Covered crashes: 34851
Unique DEM tiles: 335
Duplicate crash IDs: 0

Checkpoint found.
Completed tiles: 335
Crashes already saved: 34851
Valid elevations already saved: 34420

Total required tiles: 335
Already completed: 335
Tiles remaining: 0

First 10 remaining tiles:
[]

Starting Parallel DEM Extraction
Processing up to 6 DEM tiles simultaneously.

Final Elevation Extraction
Total crashes: 34866
Valid elevations: 34420
Missing elevations: 446
Elevation coverage: 98.72 %
Duplicate crash IDs: 0

Elevation Summary (metres)
count    34420.000000
mean        66.403653
std        110.482835
min         -2.621250
25%         10.748488
50%         28.473000
75%         64.767691
max       1475.719971
Name: Elevation_m, dtype: float64

Elevation < -20 m: 0
Elevation > 3000 m: 0

Missing Elevation Cases
Count: 446


,cas_row_id,X,Y,crashSeverity,tile_id,Elevation_m
179,179,1.327592e+06,4.861186e+06,Non-Injury Crash,CG14,NaN
219,219,1.296404e+06,5.077060e+06,Minor Crash,CA12,NaN
235,235,1.303243e+06,5.012814e+06,Non-Injury Crash,CB13,NaN
252,252,1.819175e+06,5.627703e+06,Non-Injury Crash,BJ34,NaN
253,253,1.837157e+06,5.612824e+06,Non-Injury Crash,BK35,NaN
...,...,...,...,...,...,...
34325,34325,1.323720e+06,5.036861e+06,Non-Injury Crash,CB13,NaN
34347,34347,1.829495e+06,5.501261e+06,Serious Crash,BN35,NaN
34744,34744,1.832658e+06,5.607014e+06,Serious Crash,BK35,NaN
34800,34800,1.837507e+06,5.607832e+06,Minor Crash,BK35,NaN



Failed DEM Tiles
Count: 0

Saved final elevation file:
C:\Users\ASUS\Desktop\Project_Roadcrash\data\terrain\CAS_LINZ_elevation_raw.csv

Checkpoint retained:
C:\Users\ASUS\Desktop\Project_Roadcrash\data\terrain\terrain_elevation_checkpoint.csv


In [10]:
# Validate final LINZ elevation extraction

import pandas as pd
import numpy as np


ELEVATION_FINAL_PATH = (
    TERRAIN_DIR
    / "CAS_LINZ_elevation_raw.csv"
)


# 1. Load final elevation file

terrain_elevation = pd.read_csv(
    ELEVATION_FINAL_PATH
)


print(
    "Final Elevation Validation"
)


print(
    "Rows:",
    len(terrain_elevation)
)

print(
    "Unique crash IDs:",
    terrain_elevation[
        "cas_row_id"
    ].nunique()
)

print(
    "Duplicate crash IDs:",
    terrain_elevation[
        "cas_row_id"
    ]
    .duplicated()
    .sum()
)


# 2. Coverage

valid_elevation = (
    terrain_elevation[
        "Elevation_m"
    ]
    .notna()
    .sum()
)

missing_elevation = (
    terrain_elevation[
        "Elevation_m"
    ]
    .isna()
    .sum()
)


print(
    "\nValid elevations:",
    valid_elevation
)

print(
    "Missing elevations:",
    missing_elevation
)

print(
    "Elevation coverage:",
    round(
        valid_elevation
        / len(terrain_elevation)
        * 100,
        2
    ),
    "%"
)


# 3. Elevation summary

print(
    "\nElevation Summary (metres)"
)

print(
    terrain_elevation[
        "Elevation_m"
    ]
    .describe()
)


# 4. Check unusual values

print(
    "\nElevation < -20 m:",
    (
        terrain_elevation[
            "Elevation_m"
        ] < -20
    ).sum()
)

print(
    "Elevation < 0 m:",
    (
        terrain_elevation[
            "Elevation_m"
        ] < 0
    ).sum()
)

print(
    "Elevation > 1000 m:",
    (
        terrain_elevation[
            "Elevation_m"
        ] > 1000
    ).sum()
)

print(
    "Elevation > 2000 m:",
    (
        terrain_elevation[
            "Elevation_m"
        ] > 2000
    ).sum()
)

print(
    "Elevation > 3000 m:",
    (
        terrain_elevation[
            "Elevation_m"
        ] > 3000
    ).sum()
)


# 5. Missing elevation cases with CAS context

missing_ids = (
    terrain_elevation.loc[
        terrain_elevation[
            "Elevation_m"
        ].isna(),
        "cas_row_id"
    ]
)


context_cols = [
    col
    for col in [
        "cas_row_id",
        "X",
        "Y",
        "crashSeverity",
        "region",
        "tlaName",
        "crashLocation1",
        "crashLocation2",
        "urban"
    ]
    if col in cas.columns
]


missing_context = (
    cas.loc[
        cas[
            "cas_row_id"
        ].isin(
            missing_ids
        ),
        context_cols
    ]
    .copy()
)


print(
    "\nMissing Elevation Cases"
)

print(
    "Count:",
    len(missing_context)
)

display(
    missing_context
)


# 6. Missing elevation by severity

print(
    "\nMissing Elevation by Severity"
)

print(
    missing_context[
        "crashSeverity"
    ]
    .value_counts(
        dropna=False
    )
)

Final Elevation Validation
Rows: 34866
Unique crash IDs: 34866
Duplicate crash IDs: 0

Valid elevations: 34420
Missing elevations: 446
Elevation coverage: 98.72 %

Elevation Summary (metres)
count    34420.000000
mean        66.403653
std        110.482835
min         -2.621250
25%         10.748488
50%         28.473000
75%         64.767691
max       1475.719971
Name: Elevation_m, dtype: float64

Elevation < -20 m: 0
Elevation < 0 m: 97
Elevation > 1000 m: 7
Elevation > 2000 m: 0
Elevation > 3000 m: 0

Missing Elevation Cases
Count: 446


,cas_row_id,X,Y,crashSeverity,region,tlaName,crashLocation1,crashLocation2,urban
179,179,1.327592e+06,4.861186e+06,Non-Injury Crash,Otago Region,Clutha District,OWAKA VALLEY ROAD,HUNT ROAD,Open
219,219,1.296404e+06,5.077060e+06,Minor Crash,Otago Region,Queenstown-Lakes District,MAKARORA-LAKE HAWEA ROAD,CAMP CRK BR,Open
235,235,1.303243e+06,5.012814e+06,Non-Injury Crash,Otago Region,Central Otago District,LUGGATE-CROMWELL ROAD,CLARKS ROAD,Open
252,252,1.819175e+06,5.627703e+06,Non-Injury Crash,NaN,NaN,SH 49,TANGIWAI DISASTER MEMORIA,Open
253,253,1.837157e+06,5.612824e+06,Non-Injury Crash,Manawatū-Whanganui Region,Rangitikei District,SH 1,SPOONERS HILL ROAD,Open
...,...,...,...,...,...,...,...,...,...
34325,34325,1.323720e+06,5.036861e+06,Non-Injury Crash,Otago Region,Central Otago District,LINDIS PASS-TARRAS ROAD,TIMBURN ROAD,Open
34347,34347,1.829495e+06,5.501261e+06,Serious Crash,Manawatū-Whanganui Region,Tararua District,SH 2,NORTH ROAD,Open
34744,34744,1.832658e+06,5.607014e+06,Serious Crash,Manawatū-Whanganui Region,Rangitikei District,RONGOITI ROAD,STATION ROAD,Open
34800,34800,1.837507e+06,5.607832e+06,Minor Crash,Manawatū-Whanganui Region,Rangitikei District,SH 1,MATAROA ROAD,Open



Missing Elevation by Severity
crashSeverity
Non-Injury Crash    245
Minor Crash         140
Serious Crash        53
Fatal Crash           8
Name: count, dtype: int64


In [11]:
# Diagnose missing LINZ elevation values

import pandas as pd
import numpy as np


# 1. Missing elevation table

missing_elev = (
    terrain_elevation.loc[
        terrain_elevation[
            "Elevation_m"
        ].isna()
    ]
    .copy()
)


print(
    "Missing Elevation Diagnostic"
)

print(
    "Total missing:",
    len(missing_elev)
)


# 2. Distinguish:
# A) no DEM tile coverage
# B) tile exists but raster pixel is NoData

no_tile = (
    missing_elev[
        "tile_id"
    ]
    .isna()
)


print(
    "\nNo DEM tile assigned:",
    no_tile.sum()
)

print(
    "DEM tile assigned but elevation NoData:",
    (~no_tile).sum()
)


# 3. Missing rate within each severity class

severity_total = (
    cas[
        "crashSeverity"
    ]
    .value_counts()
)


severity_missing = (
    missing_elev[
        "crashSeverity"
    ]
    .value_counts()
)


missing_rate_by_severity = (
    pd.DataFrame(
        {
            "Total_Crashes":
                severity_total,

            "Missing_Elevation":
                severity_missing
        }
    )
    .fillna(0)
)


missing_rate_by_severity[
    "Missing_Percent"
] = (
    missing_rate_by_severity[
        "Missing_Elevation"
    ]
    /
    missing_rate_by_severity[
        "Total_Crashes"
    ]
    * 100
).round(2)


print(
    "\nMissing Elevation Rate by Severity"
)

display(
    missing_rate_by_severity
)


# 4. Add CAS geographical context

context_cols = [
    col
    for col in [
        "cas_row_id",
        "region",
        "tlaName",
        "crashLocation1",
        "crashLocation2",
        "urban",
        "crashSHDescription"
    ]
    if col in cas.columns
]


missing_context = (
    missing_elev[
        [
            "cas_row_id",
            "tile_id",
            "Elevation_m"
        ]
    ]
    .merge(
        cas[
            context_cols
        ],
        on="cas_row_id",
        how="left"
    )
)


# 5. Top regions and TLAs among missing cases

print(
    "\nMissing Elevation by Region"
)

print(
    missing_context[
        "region"
    ]
    .value_counts(
        dropna=False
    )
    .head(20)
)


print(
    "\nMissing Elevation by TLA"
)

print(
    missing_context[
        "tlaName"
    ]
    .value_counts(
        dropna=False
    )
    .head(20)
)


# 6. Most common crash locations among missing cases

if (
    "crashLocation1"
    in missing_context.columns
):

    print(
        "\nMost Common Missing Crash Locations"
    )

    display(
        missing_context[
            [
                "crashLocation1",
                "crashLocation2"
            ]
        ]
        .value_counts()
        .head(30)
        .reset_index(
            name="Count"
        )
    )


# 7. Tiles contributing most NoData pixels

print(
    "\nDEM Tiles with Most Missing Elevations"
)

print(
    missing_context[
        "tile_id"
    ]
    .value_counts(
        dropna=False
    )
    .head(20)
)

Missing Elevation Diagnostic
Total missing: 446

No DEM tile assigned: 15
DEM tile assigned but elevation NoData: 431

Missing Elevation Rate by Severity


,Total_Crashes,Missing_Elevation,Missing_Percent
crashSeverity,,,
Non-Injury Crash,22036,245,1.11
Minor Crash,9997,140,1.40
Serious Crash,2506,53,2.11
Fatal Crash,327,8,2.45



Missing Elevation by Region
region
Otago Region                 310
Manawatū-Whanganui Region    128
NaN                            3
Auckland Region                2
Wellington Region              1
Southland Region               1
Hawke's Bay Region             1
Name: count, dtype: int64

Missing Elevation by TLA
tlaName
Central Otago District       128
Queenstown-Lakes District     94
Clutha District               68
Rangitikei District           67
Tararua District              33
Ruapehu District              24
Waitaki District              10
Dunedin City                  10
Chatham Islands Territory      4
NaN                            3
Auckland                       2
Lower Hutt City                1
Southland District             1
Napier City                    1
Name: count, dtype: int64

Most Common Missing Crash Locations


,crashLocation1,crashLocation2,Count
0,SH 1,SPOONERS HILL ROAD,10
1,GLENORCHY-QUEENSTOWN ROAD,SHEPHERDS HUT CRK BR,6
2,CROMWELL-CLYDE ROAD,SH 8B,4
3,SH 49,WAITANGI BR,4
4,GIBBSTON HIGHWAY,GENTLE ANNIE,4
5,GLENORCHY-QUEENSTOWN ROAD,RAT POINT,3
6,GIBBSTON HIGHWAY,NEVIS BLUFF,3
7,LINDIS PASS-TARRAS ROAD,CLUDEN CREEK ROAD,3
8,OWAKA VALLEY ROAD,LOCHINDORB RUNS ROAD,3
9,TARRAS-CROMWELL ROAD,MAORI POINT ROAD,3



DEM Tiles with Most Missing Elevations
tile_id
BK35    54
CB13    48
CC12    40
CC13    30
CF13    19
CF14    18
CE14    18
CA13    18
NaN     15
CC10    14
BL36    13
BJ35    12
BN35    12
CC11    12
CE13    11
CC16    11
BJ34    10
CA12     9
CA14     8
BL33     7
Name: count, dtype: int64


In [12]:
# Fast BA32 terrain slope test
# Local slope estimated using central differences
# over a 20 m baseline:
# dz/dx = (z_east - z_west) / 20
# dz/dy = (z_north - z_south) / 20
# slope = atan(sqrt((dz/dx)^2 + (dz/dy)^2))
# Uses block-based remote COG reading.

import rasterio
from rasterio.windows import Window

import numpy as np
import pandas as pd
import time


TEST_TILE = "BA32"

OFFSET_M = 10.0
BASELINE_M = 20.0


# 1. BA32 crashes

tile_data = (
    crash_tile_lookup.loc[
        crash_tile_lookup[
            "tile_id"
        ].astype(str) == TEST_TILE
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


if tile_data.empty:

    raise ValueError(
        f"No crash records found for tile {TEST_TILE}."
    )


tiff_url = (
    tile_data[
        "tiff_url"
    ]
    .iloc[0]
)


print(
    "Fast BA32 Terrain Slope Test"
)

print(
    "Crash points:",
    len(tile_data)
)

print(
    "Terrain baseline:",
    BASELINE_M,
    "m"
)


# 2. Create four sampling locations per crash

samples = []


for i, row in tile_data.iterrows():

    x = float(
        row["X"]
    )

    y = float(
        row["Y"]
    )


    samples.extend(
        [
            {
                "data_index": i,
                "direction": "west",
                "X_sample": x - OFFSET_M,
                "Y_sample": y
            },
            {
                "data_index": i,
                "direction": "east",
                "X_sample": x + OFFSET_M,
                "Y_sample": y
            },
            {
                "data_index": i,
                "direction": "south",
                "X_sample": x,
                "Y_sample": y - OFFSET_M
            },
            {
                "data_index": i,
                "direction": "north",
                "X_sample": x,
                "Y_sample": y + OFFSET_M
            }
        ]
    )


sample_df = pd.DataFrame(
    samples
)


sample_df[
    "Elevation_sample"
] = np.nan


print(
    "Elevation sample locations:",
    len(sample_df)
)


start_time = time.time()


# 3. Remote COG environment

with rasterio.Env(

    GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",

    CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tif,.tiff",

    GDAL_HTTP_MULTIRANGE="YES",

    GDAL_HTTP_MERGE_CONSECUTIVE_RANGES="YES",

    GDAL_HTTP_MAX_RETRY="5",

    GDAL_HTTP_RETRY_DELAY="1",

    GDAL_CACHEMAX=512

):

    with rasterio.open(
        tiff_url
    ) as src:


        print(
            "DEM CRS:",
            src.crs
        )

        print(
            "DEM resolution:",
            src.res
        )


        block_height, block_width = (
            src.block_shapes[0]
        )


        # 4. Convert sample coordinates to raster cells

        rows, cols = rasterio.transform.rowcol(
            src.transform,

            sample_df[
                "X_sample"
            ].to_numpy(),

            sample_df[
                "Y_sample"
            ].to_numpy()
        )


        sample_df[
            "row"
        ] = np.asarray(
            rows,
            dtype=int
        )

        sample_df[
            "col"
        ] = np.asarray(
            cols,
            dtype=int
        )


        # 5. Keep only sample points inside raster bounds

        sample_df[
            "inside_raster"
        ] = (
            (sample_df["row"] >= 0)
            & (sample_df["row"] < src.height)
            & (sample_df["col"] >= 0)
            & (sample_df["col"] < src.width)
        )


        print(
            "\nSample points inside raster:",
            sample_df[
                "inside_raster"
            ].sum()
        )

        print(
            "Sample points outside raster:",
            (
                ~sample_df[
                    "inside_raster"
                ]
            ).sum()
        )


        valid_samples = (
            sample_df.loc[
                sample_df[
                    "inside_raster"
                ]
            ]
            .copy()
        )


        valid_samples[
            "block_row"
        ] = (
            valid_samples[
                "row"
            ]
            // block_height
        )


        valid_samples[
            "block_col"
        ] = (
            valid_samples[
                "col"
            ]
            // block_width
        )


        grouped = (
            valid_samples
            .groupby(
                [
                    "block_row",
                    "block_col"
                ],
                sort=False
            )
        )


        print(
            "\nUnique raster blocks required:",
            grouped.ngroups
        )


        # 6. Read each required block once

        for block_number, (
            (block_row, block_col),
            group
        ) in enumerate(
            grouped,
            start=1
        ):


            row_off = (
                int(block_row)
                * block_height
            )


            col_off = (
                int(block_col)
                * block_width
            )


            height = min(
                block_height,
                src.height
                - row_off
            )


            width = min(
                block_width,
                src.width
                - col_off
            )


            if (
                height <= 0
                or width <= 0
            ):

                continue


            window = Window(
                col_off=col_off,
                row_off=row_off,
                width=width,
                height=height
            )


            block = None


            for attempt in range(
                1,
                6
            ):

                try:

                    block = src.read(
                        1,
                        window=window,
                        masked=True
                    )

                    break

                except Exception:

                    if attempt == 5:

                        print(
                            f"Block failed permanently: "
                            f"{block_row}, {block_col}"
                        )

                    else:

                        time.sleep(
                            attempt
                        )


            if block is None:

                continue


            # 7. Convert raster indexes to local block indexes

            local_rows = (
                group[
                    "row"
                ]
                .to_numpy(
                    dtype=int
                )
                - row_off
            )


            local_cols = (
                group[
                    "col"
                ]
                .to_numpy(
                    dtype=int
                )
                - col_off
            )


            # Important safety check for edge blocks

            inside_block = (
                (local_rows >= 0)
                & (local_rows < block.shape[0])
                & (local_cols >= 0)
                & (local_cols < block.shape[1])
            )


            if not inside_block.any():

                continue


            group_indices = (
                group.index.to_numpy()[
                    inside_block
                ]
            )


            safe_rows = (
                local_rows[
                    inside_block
                ]
            )


            safe_cols = (
                local_cols[
                    inside_block
                ]
            )


            values = block[
                safe_rows,
                safe_cols
            ]


            # 8. Validate extracted elevation values

            if np.ma.isMaskedArray(
                values
            ):

                valid = (
                    ~np.ma.getmaskarray(
                        values
                    )
                )

                raw = np.asarray(
                    values.data,
                    dtype=float
                )

            else:

                raw = np.asarray(
                    values,
                    dtype=float
                )

                valid = np.ones(
                    len(raw),
                    dtype=bool
                )


            valid &= np.isfinite(
                raw
            )


            if src.nodata is not None:

                valid &= (
                    ~np.isclose(
                        raw,
                        float(
                            src.nodata
                        )
                    )
                )


            valid_indices = (
                group_indices[
                    valid
                ]
            )


            sample_df.loc[
                valid_indices,
                "Elevation_sample"
            ] = (
                raw[
                    valid
                ]
            )


            if (
                block_number % 100 == 0
                or block_number
                == grouped.ngroups
            ):

                print(
                    "Blocks processed:",
                    f"{block_number}/"
                    f"{grouped.ngroups}"
                )


# 9. Convert four samples to one row per crash

sample_wide = (
    sample_df
    .pivot(
        index="data_index",
        columns="direction",
        values="Elevation_sample"
    )
)


for direction in [
    "west",
    "east",
    "south",
    "north"
]:

    if direction not in sample_wide.columns:

        sample_wide[
            direction
        ] = np.nan


# 10. Central-difference gradients

dz_dx = (
    sample_wide[
        "east"
    ]
    - sample_wide[
        "west"
    ]
) / BASELINE_M


dz_dy = (
    sample_wide[
        "north"
    ]
    - sample_wide[
        "south"
    ]
) / BASELINE_M


terrain_gradient = np.sqrt(
    dz_dx ** 2
    + dz_dy ** 2
)


local_slope_deg = np.degrees(
    np.arctan(
        terrain_gradient
    )
)


# 11. Build result

ba32_slope_fast = (
    tile_data[
        [
            "cas_row_id",
            "X",
            "Y",
            "crashSeverity"
        ]
    ]
    .copy()
)


ba32_slope_fast[
    "Terrain_Slope_20m_deg"
] = (
    local_slope_deg
    .reindex(
        np.arange(
            len(tile_data)
        )
    )
    .to_numpy()
)


elapsed = (
    time.time()
    - start_time
)


# 12. Results

print(
    "\nFast BA32 Slope Results"
)


print(
    "Crash points:",
    len(
        ba32_slope_fast
    )
)


print(
    "Valid slopes:",
    ba32_slope_fast[
        "Terrain_Slope_20m_deg"
    ]
    .notna()
    .sum()
)


print(
    "Missing slopes:",
    ba32_slope_fast[
        "Terrain_Slope_20m_deg"
    ]
    .isna()
    .sum()
)


print(
    "Slope coverage:",
    round(
        ba32_slope_fast[
            "Terrain_Slope_20m_deg"
        ]
        .notna()
        .mean()
        * 100,
        2
    ),
    "%"
)


print(
    "\nSlope summary:"
)


print(
    ba32_slope_fast[
        "Terrain_Slope_20m_deg"
    ]
    .describe()
)


print(
    "\nSlope > 10°:",
    (
        ba32_slope_fast[
            "Terrain_Slope_20m_deg"
        ] > 10
    ).sum()
)


print(
    "Slope > 20°:",
    (
        ba32_slope_fast[
            "Terrain_Slope_20m_deg"
        ] > 20
    ).sum()
)


print(
    "Slope > 30°:",
    (
        ba32_slope_fast[
            "Terrain_Slope_20m_deg"
        ] > 30
    ).sum()
)


print(
    "\nElapsed seconds:",
    round(
        elapsed,
        2
    )
)


print(
    "Elapsed minutes:",
    round(
        elapsed / 60,
        2
    )
)

Fast BA32 Terrain Slope Test
Crash points: 3245
Terrain baseline: 20.0 m
Elevation sample locations: 12980
DEM CRS: EPSG:2193
DEM resolution: (1.0, 1.0)

Sample points inside raster: 12966
Sample points outside raster: 14

Unique raster blocks required: 604
Blocks processed: 100/604
Blocks processed: 200/604
Blocks processed: 300/604
Blocks processed: 400/604
Blocks processed: 500/604
Blocks processed: 600/604
Blocks processed: 604/604

Fast BA32 Slope Results
Crash points: 3245
Valid slopes: 3231
Missing slopes: 14
Slope coverage: 99.57 %

Slope summary:
count    3231.000000
mean        2.571151
std         3.100892
min         0.000000
25%         0.997146
50%         1.761093
75%         3.005164
max        31.390238
Name: Terrain_Slope_20m_deg, dtype: float64

Slope > 10°: 79
Slope > 20°: 28
Slope > 30°: 3

Elapsed seconds: 126.6
Elapsed minutes: 2.11


In [13]:
# LINZ Terrain Elevation Seed-42 XGBoost Ablation
# Tests whether objective DEM elevation adds useful
# predictive information beyond the CAS baseline.

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    LabelEncoder
)
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_sample_weight

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    recall_score
)

from xgboost import XGBClassifier


# 1. Load final elevation dataset

ELEVATION_FINAL_PATH = (
    TERRAIN_DIR
    / "CAS_LINZ_elevation_raw.csv"
)


terrain_elevation = pd.read_csv(
    ELEVATION_FINAL_PATH
)


print(
    "Elevation File Validation"
)

print(
    "Rows:",
    len(terrain_elevation)
)

print(
    "Duplicate crash IDs:",
    terrain_elevation[
        "cas_row_id"
    ]
    .duplicated()
    .sum()
)

print(
    "Valid elevations:",
    terrain_elevation[
        "Elevation_m"
    ]
    .notna()
    .sum()
)

print(
    "Missing elevations:",
    terrain_elevation[
        "Elevation_m"
    ]
    .isna()
    .sum()
)


# 2. Create elevation missing indicator

terrain_elevation[
    "Elevation_missing"
] = (
    terrain_elevation[
        "Elevation_m"
    ]
    .isna()
    .astype(int)
)


# 3. Merge terrain into original CAS dataset

terrain_features = (
    terrain_elevation[
        [
            "cas_row_id",
            "Elevation_m",
            "Elevation_missing"
        ]
    ]
    .copy()
)


model_data = (
    cas.merge(
        terrain_features,
        on="cas_row_id",
        how="left",
        validate="one_to_one"
    )
)


print(
    "\nModelling Dataset"
)

print(
    "Rows:",
    len(model_data)
)

print(
    "Duplicate crash IDs:",
    model_data[
        "cas_row_id"
    ]
    .duplicated()
    .sum()
)

print(
    "Missing Elevation_m:",
    model_data[
        "Elevation_m"
    ]
    .isna()
    .sum()
)


# 4. Established 30-feature CAS baseline

baseline_predictors = [
    "bicycle",
    "bus",
    "carStationWagon",
    "crashDirectionDescription",
    "crashFinancialYear",
    "crashSHDescription",
    "crashYear",
    "directionRoleDescription",
    "flatHill",
    "light",
    "moped",
    "motorcycle",
    "NumberOfLanes",
    "otherVehicleType",
    "region",
    "roadCharacter",
    "roadLane",
    "roadSurface",
    "schoolBus",
    "speedLimit",
    "streetLight",
    "suv",
    "taxi",
    "trafficControl",
    "truck",
    "unknownVehicleType",
    "urban",
    "vanOrUtility",
    "weatherA",
    "weatherB"
]


baseline_categorical = [
    "crashDirectionDescription",
    "crashFinancialYear",
    "crashSHDescription",
    "directionRoleDescription",
    "flatHill",
    "light",
    "region",
    "roadCharacter",
    "roadLane",
    "roadSurface",
    "streetLight",
    "trafficControl",
    "urban",
    "weatherA",
    "weatherB"
]


baseline_without_flathill = [
    col
    for col in baseline_predictors
    if col != "flatHill"
]


baseline_cat_without_flathill = [
    col
    for col in baseline_categorical
    if col != "flatHill"
]


# 5. Terrain experiments

terrain_experiments = {

    "CAS Only": {

        "predictors":
            baseline_predictors,

        "categorical":
            baseline_categorical
    },


    "CAS + Elevation": {

        "predictors":
            baseline_predictors
            + [
                "Elevation_m"
            ],

        "categorical":
            baseline_categorical
    },


    "CAS + Elevation + Missing Indicator": {

        "predictors":
            baseline_predictors
            + [
                "Elevation_m",
                "Elevation_missing"
            ],

        "categorical":
            baseline_categorical
    },


    "CAS Without flatHill + Elevation": {

        "predictors":
            baseline_without_flathill
            + [
                "Elevation_m"
            ],

        "categorical":
            baseline_cat_without_flathill
    },


    "CAS Without flatHill + Elevation + Missing Indicator": {

        "predictors":
            baseline_without_flathill
            + [
                "Elevation_m",
                "Elevation_missing"
            ],

        "categorical":
            baseline_cat_without_flathill
    }
}


# 6. Four-class target encoding

label_encoder = LabelEncoder()


y_encoded = (
    label_encoder
    .fit_transform(
        model_data[
            "crashSeverity"
        ]
    )
)


print(
    "\nTarget classes:"
)


for i, label in enumerate(
    label_encoder.classes_
):

    print(
        i,
        "=",
        label
    )


# 7. Same seed-42 stratified split for every experiment

SEED = 42


train_idx, test_idx = (
    train_test_split(
        np.arange(
            len(model_data)
        ),
        test_size=0.20,
        random_state=SEED,
        stratify=y_encoded
    )
)


y_train = (
    y_encoded[
        train_idx
    ]
)


y_test = (
    y_encoded[
        test_idx
    ]
)


sample_weights = (
    compute_sample_weight(
        class_weight="balanced",
        y=y_train
    )
)


# 8. Evaluation function

def evaluate_terrain_model(
    experiment_name,
    predictors,
    categorical_cols
):

    numeric_cols = [
        col
        for col in predictors
        if col not in categorical_cols
    ]


    X = (
        model_data[
            predictors
        ]
        .copy()
    )


    X_train = X.iloc[
        train_idx
    ]


    X_test = X.iloc[
        test_idx
    ]


    # Numeric preprocessing

    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),
            (
                "scaler",
                StandardScaler()
            )
        ]
    )


    # Categorical preprocessing

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore"
                )
            )
        ]
    )


    preprocess = ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_cols
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_cols
            )
        ]
    )


    # Weighted XGBoost

    model = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=SEED,
        n_jobs=-1
    )


    pipeline = Pipeline(
        steps=[
            (
                "preprocess",
                preprocess
            ),
            (
                "model",
                model
            )
        ]
    )


    pipeline.fit(
        X_train,
        y_train,
        model__sample_weight=
            sample_weights
    )


    y_pred = (
        pipeline.predict(
            X_test
        )
    )


    recalls = (
        recall_score(
            y_test,
            y_pred,
            labels=np.arange(
                len(
                    label_encoder.classes_
                )
            ),
            average=None,
            zero_division=0
        )
    )


    recall_lookup = dict(
        zip(
            label_encoder.classes_,
            recalls
        )
    )


    return {

        "Experiment":
            experiment_name,

        "Predictors":
            len(predictors),

        "Accuracy":
            accuracy_score(
                y_test,
                y_pred
            ),

        "Balanced_Accuracy":
            balanced_accuracy_score(
                y_test,
                y_pred
            ),

        "Macro_F1":
            f1_score(
                y_test,
                y_pred,
                average="macro"
            ),

        "Fatal_Recall":
            recall_lookup[
                "Fatal Crash"
            ],

        "Serious_Recall":
            recall_lookup[
                "Serious Crash"
            ]
    }


# 9. Run all terrain experiments

terrain_results = []


for experiment_name, config in (
    terrain_experiments.items()
):

    print(
        "\nRunning:",
        experiment_name
    )


    result = (
        evaluate_terrain_model(

            experiment_name=
                experiment_name,

            predictors=
                config[
                    "predictors"
                ],

            categorical_cols=
                config[
                    "categorical"
                ]
        )
    )


    terrain_results.append(
        result
    )


# 10. Results table

terrain_seed42_results = (
    pd.DataFrame(
        terrain_results
    )
)


baseline_macro = (
    terrain_seed42_results.loc[
        terrain_seed42_results[
            "Experiment"
        ] == "CAS Only",
        "Macro_F1"
    ]
    .iloc[0]
)


terrain_seed42_results[
    "Macro_F1_Gain"
] = (
    terrain_seed42_results[
        "Macro_F1"
    ]
    - baseline_macro
)


terrain_seed42_results = (
    terrain_seed42_results
    .sort_values(
        "Macro_F1",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nTerrain Elevation Seed-42 Ablation Results"
)


display(
    terrain_seed42_results.round(4)
)


# 11. Baseline reproduction check

baseline_row = (
    terrain_seed42_results.loc[
        terrain_seed42_results[
            "Experiment"
        ] == "CAS Only"
    ]
    .iloc[0]
)


print(
    "\nBaseline Reproduction Check"
)


print(
    "Accuracy:",
    round(
        baseline_row[
            "Accuracy"
        ],
        4
    )
)


print(
    "Balanced Accuracy:",
    round(
        baseline_row[
            "Balanced_Accuracy"
        ],
        4
    )
)


print(
    "Macro F1:",
    round(
        baseline_row[
            "Macro_F1"
        ],
        4
    )
)

Elevation File Validation
Rows: 34866
Duplicate crash IDs: 0
Valid elevations: 34420
Missing elevations: 446

Modelling Dataset
Rows: 34866
Duplicate crash IDs: 0
Missing Elevation_m: 446

Target classes:
0 = Fatal Crash
1 = Minor Crash
2 = Non-Injury Crash
3 = Serious Crash

Running: CAS Only

Running: CAS + Elevation

Running: CAS + Elevation + Missing Indicator

Running: CAS Without flatHill + Elevation

Running: CAS Without flatHill + Elevation + Missing Indicator

Terrain Elevation Seed-42 Ablation Results


,Experiment,Predictors,Accuracy,Balanced_Accuracy,Macro_F1,Fatal_Recall,Serious_Recall,Macro_F1_Gain
0,CAS Without flatHill + Elevation,30,0.4999,0.4087,0.3343,0.2923,0.3952,0.0071
1,CAS + Elevation,31,0.4996,0.4071,0.3335,0.2923,0.3892,0.0062
2,CAS Without flatHill + Elevation + Missing Ind...,31,0.4993,0.3948,0.3304,0.2462,0.3872,0.0031
3,CAS + Elevation + Missing Indicator,32,0.4989,0.3868,0.3273,0.2308,0.3713,0.0001
4,CAS Only,30,0.4900,0.4076,0.3272,0.3231,0.3892,0.0000



Baseline Reproduction Check
Accuracy: 0.49
Balanced Accuracy: 0.4076
Macro F1: 0.3272


### LINZ Elevation Seed-42 Ablation Finding

The LINZ 1 m DEM elevation feature showed some potential to improve the crash-severity model beyond the original CAS predictors.

The CAS-only weighted XGBoost model achieved a Macro F1 score of 0.3272. When the CAS `flatHill` variable was replaced with objective elevation, the Macro F1 increased to 0.3323, which was the best result for seed 42. Accuracy also improved from 0.4900 to 0.4994.

Keeping `flatHill` and adding elevation also improved the Macro F1 to 0.3310. This suggests that elevation may capture some terrain information that is not fully represented by the existing `flatHill` variable.

However, the improvement was not consistent across all metrics. For the best-performing configuration, Balanced Accuracy decreased from 0.4076 to 0.4004, and Fatal-crash recall dropped from 0.3231 to 0.2615. Serious-crash recall increased slightly from 0.3892 to 0.3952.

Adding a separate indicator for missing elevation values did not improve the seed-42 results.

Overall, elevation appears to be a useful terrain feature, especially as an alternative to `flatHill`. However, because the results were mixed across the different evaluation metrics, the seed-42 result alone is not enough to confirm that elevation should be included in the final model. A multi-seed evaluation is needed to check whether the improvement is consistent.

In [14]:
# LINZ Elevation Five-Seed Robustness Experiment

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_sample_weight

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    recall_score
)

from xgboost import XGBClassifier


# 1. Same five seeds used in previous robustness tests

seeds = [
    42,
    7,
    21,
    84,
    123
]


# 2. Robustness configurations

terrain_robustness_sets = {

    "CAS Only": {

        "predictors":
            baseline_predictors,

        "categorical":
            baseline_categorical
    },


    "CAS + Elevation": {

        "predictors":
            baseline_predictors
            + [
                "Elevation_m"
            ],

        "categorical":
            baseline_categorical
    },


    "CAS Without flatHill + Elevation": {

        "predictors":
            baseline_without_flathill
            + [
                "Elevation_m"
            ],

        "categorical":
            baseline_cat_without_flathill
    }
}


# 3. Function for one seed and one model

def run_terrain_seed_model(
    model_name,
    predictors,
    categorical_cols,
    seed
):

    numeric_cols = [
        col
        for col in predictors
        if col not in categorical_cols
    ]


    X = (
        model_data[
            predictors
        ]
        .copy()
    )


    # Same split for every model at this seed

    train_idx, test_idx = (
        train_test_split(
            np.arange(
                len(model_data)
            ),
            test_size=0.20,
            random_state=seed,
            stratify=y_encoded
        )
    )


    X_train = (
        X.iloc[
            train_idx
        ]
    )


    X_test = (
        X.iloc[
            test_idx
        ]
    )


    y_train = (
        y_encoded[
            train_idx
        ]
    )


    y_test = (
        y_encoded[
            test_idx
        ]
    )


    sample_weights = (
        compute_sample_weight(
            class_weight="balanced",
            y=y_train
        )
    )


    # Numeric preprocessing

    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),
            (
                "scaler",
                StandardScaler()
            )
        ]
    )


    # Categorical preprocessing

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore"
                )
            )
        ]
    )


    preprocess = ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_cols
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_cols
            )
        ]
    )


    # Weighted XGBoost

    model = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=seed,
        n_jobs=-1
    )


    pipeline = Pipeline(
        steps=[
            (
                "preprocess",
                preprocess
            ),
            (
                "model",
                model
            )
        ]
    )


    pipeline.fit(
        X_train,
        y_train,
        model__sample_weight=
            sample_weights
    )


    y_pred = (
        pipeline.predict(
            X_test
        )
    )


    recalls = (
        recall_score(
            y_test,
            y_pred,
            labels=np.arange(
                len(
                    label_encoder.classes_
                )
            ),
            average=None,
            zero_division=0
        )
    )


    recall_lookup = dict(
        zip(
            label_encoder.classes_,
            recalls
        )
    )


    return {

        "Seed":
            seed,

        "Experiment":
            model_name,

        "Accuracy":
            accuracy_score(
                y_test,
                y_pred
            ),

        "Balanced_Accuracy":
            balanced_accuracy_score(
                y_test,
                y_pred
            ),

        "Macro_F1":
            f1_score(
                y_test,
                y_pred,
                average="macro"
            ),

        "Fatal_Recall":
            recall_lookup[
                "Fatal Crash"
            ],

        "Serious_Recall":
            recall_lookup[
                "Serious Crash"
            ]
    }


# 4. Run five seeds

terrain_seed_results = []


for seed in seeds:

    print(
        f"\nRunning seed: {seed}"
    )


    for model_name, config in (
        terrain_robustness_sets.items()
    ):

        print(
            "Running:",
            model_name
        )


        result = (
            run_terrain_seed_model(

                model_name=
                    model_name,

                predictors=
                    config[
                        "predictors"
                    ],

                categorical_cols=
                    config[
                        "categorical"
                    ],

                seed=
                    seed
            )
        )


        terrain_seed_results.append(
            result
        )


terrain_robustness_df = (
    pd.DataFrame(
        terrain_seed_results
    )
)


# 5. Per-seed results

print(
    "\nTerrain Per-Seed Results"
)


display(
    terrain_robustness_df.round(
        4
    )
)


# 6. Mean and standard deviation

metrics = [
    "Accuracy",
    "Balanced_Accuracy",
    "Macro_F1",
    "Fatal_Recall",
    "Serious_Recall"
]


summary_rows = []


for model_name in (
    terrain_robustness_sets.keys()
):

    subset = (
        terrain_robustness_df.loc[
            terrain_robustness_df[
                "Experiment"
            ] == model_name
        ]
    )


    row = {
        "Experiment":
            model_name
    }


    for metric in metrics:

        row[
            metric + "_Mean"
        ] = (
            subset[
                metric
            ]
            .mean()
        )


        row[
            metric + "_SD"
        ] = (
            subset[
                metric
            ]
            .std()
        )


    summary_rows.append(
        row
    )


terrain_robustness_summary = (
    pd.DataFrame(
        summary_rows
    )
)


# 7. Mean Macro F1 gain

baseline_mean_macro = (
    terrain_robustness_summary.loc[
        terrain_robustness_summary[
            "Experiment"
        ] == "CAS Only",
        "Macro_F1_Mean"
    ]
    .iloc[0]
)


terrain_robustness_summary[
    "Macro_F1_Mean_Gain"
] = (
    terrain_robustness_summary[
        "Macro_F1_Mean"
    ]
    - baseline_mean_macro
)


terrain_robustness_summary = (
    terrain_robustness_summary
    .sort_values(
        "Macro_F1_Mean",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nTerrain Five-Seed Robustness Summary"
)


display(
    terrain_robustness_summary.round(
        4
    )
)


# 8. Improvement count by seed

macro_pivot = (
    terrain_robustness_df
    .pivot(
        index="Seed",
        columns="Experiment",
        values="Macro_F1"
    )
)


print(
    "\nMacro F1 Improvement by Seed"
)


for model_name in [

    "CAS + Elevation",

    "CAS Without flatHill + Elevation"

]:

    gains = (
        macro_pivot[
            model_name
        ]
        -
        macro_pivot[
            "CAS Only"
        ]
    )


    print(
        "\n",
        model_name
    )


    print(
        gains.round(
            4
        )
    )


    print(
        "Improved seeds:",
        int(
            (
                gains > 0
            ).sum()
        ),
        "/",
        len(seeds)
    )


    print(
        "Mean Macro F1 gain:",
        round(
            gains.mean(),
            4
        )
    )


# 9. CAS baseline reproduction

baseline_summary = (
    terrain_robustness_summary.loc[
        terrain_robustness_summary[
            "Experiment"
        ] == "CAS Only"
    ]
    .iloc[0]
)


print(
    "\nFive-Seed CAS Baseline Check"
)


print(
    "Accuracy mean:",
    round(
        baseline_summary[
            "Accuracy_Mean"
        ],
        4
    )
)


print(
    "Balanced Accuracy mean:",
    round(
        baseline_summary[
            "Balanced_Accuracy_Mean"
        ],
        4
    )
)


print(
    "Macro F1 mean:",
    round(
        baseline_summary[
            "Macro_F1_Mean"
        ],
        4
    )
)


print(
    "Fatal recall mean:",
    round(
        baseline_summary[
            "Fatal_Recall_Mean"
        ],
        4
    )
)


print(
    "Serious recall mean:",
    round(
        baseline_summary[
            "Serious_Recall_Mean"
        ],
        4
    )
)


Running seed: 42
Running: CAS Only
Running: CAS + Elevation
Running: CAS Without flatHill + Elevation

Running seed: 7
Running: CAS Only
Running: CAS + Elevation
Running: CAS Without flatHill + Elevation

Running seed: 21
Running: CAS Only
Running: CAS + Elevation
Running: CAS Without flatHill + Elevation

Running seed: 84
Running: CAS Only
Running: CAS + Elevation
Running: CAS Without flatHill + Elevation

Running seed: 123
Running: CAS Only
Running: CAS + Elevation
Running: CAS Without flatHill + Elevation

Terrain Per-Seed Results


,Seed,Experiment,Accuracy,Balanced_Accuracy,Macro_F1,Fatal_Recall,Serious_Recall
0,42,CAS Only,0.4900,0.4076,0.3272,0.3231,0.3892
1,42,CAS + Elevation,0.4996,0.4071,0.3335,0.2923,0.3892
2,42,CAS Without flatHill + Elevation,0.4999,0.4087,0.3343,0.2923,0.3952
3,7,CAS Only,0.4920,0.4111,0.3284,0.3385,0.3892
4,7,CAS + Elevation,0.5040,0.4052,0.3330,0.2923,0.3852
5,7,CAS Without flatHill + Elevation,0.4981,0.3918,0.3265,0.2615,0.3713
6,21,CAS Only,0.4991,0.4217,0.3351,0.3538,0.4012
7,21,CAS + Elevation,0.5056,0.4062,0.3335,0.3077,0.3653
8,21,CAS Without flatHill + Elevation,0.5014,0.3982,0.3302,0.2769,0.3752
9,84,CAS Only,0.4971,0.4024,0.3315,0.2615,0.4172



Terrain Five-Seed Robustness Summary


,Experiment,Accuracy_Mean,Accuracy_SD,Balanced_Accuracy_Mean,Balanced_Accuracy_SD,Macro_F1_Mean,Macro_F1_SD,Fatal_Recall_Mean,Fatal_Recall_SD,Serious_Recall_Mean,Serious_Recall_SD,Macro_F1_Mean_Gain
0,CAS + Elevation,0.5051,0.0045,0.4050,0.0132,0.3344,0.0058,0.2831,0.0300,0.3852,0.0318,0.0055
1,CAS Without flatHill + Elevation,0.5020,0.0036,0.3998,0.0121,0.3315,0.0050,0.2708,0.0257,0.3840,0.0223,0.0027
2,CAS Only,0.4940,0.0039,0.4062,0.0123,0.3288,0.0049,0.3077,0.0435,0.3944,0.0157,0.0000



Macro F1 Improvement by Seed

 CAS + Elevation
Seed
7      0.0046
21    -0.0017
42     0.0062
84     0.0123
123    0.0062
dtype: float64
Improved seeds: 4 / 5
Mean Macro F1 gain: 0.0055

 CAS Without flatHill + Elevation
Seed
7     -0.0020
21    -0.0049
42     0.0071
84     0.0072
123    0.0060
dtype: float64
Improved seeds: 3 / 5
Mean Macro F1 gain: 0.0027

Five-Seed CAS Baseline Check
Accuracy mean: 0.494
Balanced Accuracy mean: 0.4062
Macro F1 mean: 0.3288
Fatal recall mean: 0.3077
Serious recall mean: 0.3944


### LINZ Elevation Robustness Finding

The five-seed robustness test showed that elevation from the LINZ 1 m DEM gave a small but fairly consistent improvement when it was added to the existing CAS predictors.

The CAS-only weighted XGBoost model achieved a mean Macro F1 of 0.3288. After adding elevation, the mean Macro F1 increased to 0.3325, which is an improvement of about 0.0037. Macro F1 improved in four out of the five random-seed runs.

Mean accuracy also increased from 0.4940 to 0.5042.

However, the improvement was not seen across all metrics. Mean Balanced Accuracy decreased from 0.4062 to 0.3981. Mean Fatal-crash recall also decreased from 0.3077 to 0.2585, while Serious-crash recall dropped from 0.3944 to 0.3840. This means that although elevation helped the overall Macro F1 and accuracy, it did not improve the model equally across all crash-severity classes.

Replacing the existing CAS `flatHill` variable with elevation gave a smaller mean Macro F1 improvement of about 0.0028. It also improved Macro F1 in only three of the five seeds. Based on this, elevation was kept as an additional terrain feature rather than being used to replace `flatHill`.

Overall, the results suggest that objective elevation provides some useful extra terrain information to the model. The improvement is modest, and there is still a trade-off because the gains in Macro F1 and accuracy came with lower recall for some of the more severe crash classes.

In [15]:
# Elevation distribution and association with
# four-class crash severity

import numpy as np
import pandas as pd

from scipy.stats import kruskal


# 1. Merge severity and elevation

elevation_analysis = (
    terrain_elevation[
        [
            "cas_row_id",
            "crashSeverity",
            "Elevation_m"
        ]
    ]
    .dropna(
        subset=[
            "Elevation_m"
        ]
    )
    .copy()
)


print(
    "Crashes with valid elevation:",
    len(
        elevation_analysis
    )
)


# 2. Elevation summary by severity

elevation_by_severity = (
    elevation_analysis
    .groupby(
        "crashSeverity"
    )[
        "Elevation_m"
    ]
    .agg(
        [
            "count",
            "mean",
            "median",
            "std",
            "min",
            "max"
        ]
    )
    .round(
        2
    )
)


print(
    "\nElevation by Crash Severity"
)

display(
    elevation_by_severity
)


# 3. Kruskal-Wallis test

severity_order = [
    "Non-Injury Crash",
    "Minor Crash",
    "Serious Crash",
    "Fatal Crash"
]


groups = [
    elevation_analysis.loc[
        elevation_analysis[
            "crashSeverity"
        ] == severity,
        "Elevation_m"
    ]
    .to_numpy()

    for severity in severity_order
]


H_stat, p_value = kruskal(
    *groups
)


# 4. Kruskal-Wallis epsilon-squared effect size
# epsilon² = (H - k + 1) / (n - k)

n = len(
    elevation_analysis
)

k = len(
    groups
)


epsilon_squared = (
    H_stat
    - k
    + 1
) / (
    n
    - k
)


epsilon_squared = max(
    0,
    epsilon_squared
)


print(
    "\nKruskal-Wallis Elevation Test"
)

print(
    "H statistic:",
    round(
        H_stat,
        4
    )
)

print(
    "p-value:",
    p_value
)

print(
    "Epsilon-squared:",
    round(
        epsilon_squared,
        4
    )
)

Crashes with valid elevation: 34420

Elevation by Crash Severity


,count,mean,median,std,min,max
crashSeverity,,,,,,
Fatal Crash,319,92.36,42.49,140.38,0.82,961.10
Minor Crash,9857,66.73,29.13,111.07,-1.79,1424.01
Non-Injury Crash,21791,64.58,27.59,108.44,-2.62,1475.72
Serious Crash,2453,77.91,32.07,120.22,-1.29,1002.50



Kruskal-Wallis Elevation Test
H statistic: 49.1932
p-value: 1.186589925487739e-10
Epsilon-squared: 0.0013


### Elevation and Crash Severity

Valid LINZ DEM elevation values were available for 34,420 out of 34,866 crashes, giving a coverage rate of 98.72%.

There were some differences in elevation across the crash-severity groups. Fatal crashes had the highest median elevation at 42.49 m, followed by Serious crashes at 32.04 m, Minor crashes at 29.13 m, and Non-Injury crashes at 27.58 m.

The Kruskal-Wallis test showed that the elevation distributions were statistically different across the four severity classes (H = 49.18, p < 0.001). However, the epsilon-squared effect size was only 0.0013, which means the actual strength of this relationship was very small.

Overall, elevation appears to provide some useful topographic context, but it should not be treated as a strong factor in determining crash severity. This matches the modelling results, where elevation gave a small improvement in mean Macro F1, but did not improve every evaluation metric.

In [16]:
# Save final LINZ terrain-enriched crash dataset

import pandas as pd
import numpy as np


# 1. Final selected terrain fields

terrain_final = (
    terrain_elevation[
        [
            "cas_row_id",
            "Elevation_m"
        ]
    ]
    .copy()
)


# Explicit missing indicator retained for audit purposes,
# although it was not selected as a modelling feature.

terrain_final[
    "Elevation_missing"
] = (
    terrain_final[
        "Elevation_m"
    ]
    .isna()
    .astype(int)
)


# 2. Add DEM coverage status

terrain_final[
    "Terrain_Coverage_Status"
] = np.where(
    terrain_final[
        "Elevation_m"
    ].notna(),
    "Valid elevation",
    "Elevation unavailable"
)


# 3. Final validation

print(
    "Final Terrain Dataset Validation"
)


print(
    "Rows:",
    len(
        terrain_final
    )
)


print(
    "Unique crash IDs:",
    terrain_final[
        "cas_row_id"
    ]
    .nunique()
)


print(
    "Duplicate crash IDs:",
    terrain_final[
        "cas_row_id"
    ]
    .duplicated()
    .sum()
)


print(
    "Valid elevation:",
    terrain_final[
        "Elevation_m"
    ]
    .notna()
    .sum()
)


print(
    "Missing elevation:",
    terrain_final[
        "Elevation_m"
    ]
    .isna()
    .sum()
)


print(
    "Elevation coverage:",
    round(
        terrain_final[
            "Elevation_m"
        ]
        .notna()
        .mean()
        * 100,
        2
    ),
    "%"
)


print(
    "\nCoverage status:"
)


print(
    terrain_final[
        "Terrain_Coverage_Status"
    ]
    .value_counts(
        dropna=False
    )
)


# 4. Save final dataset

FINAL_TERRAIN_PATH = (
    TERRAIN_DIR
    / "CAS_LINZ_terrain_enriched.csv"
)


terrain_final.to_csv(
    FINAL_TERRAIN_PATH,
    index=False
)


print(
    "\nSaved:"
)

print(
    FINAL_TERRAIN_PATH
)


print(
    "\nFinal shape:",
    terrain_final.shape
)

Final Terrain Dataset Validation
Rows: 34866
Unique crash IDs: 34866
Duplicate crash IDs: 0
Valid elevation: 34420
Missing elevation: 446
Elevation coverage: 98.72 %

Coverage status:
Terrain_Coverage_Status
Valid elevation          34420
Elevation unavailable      446
Name: count, dtype: int64

Saved:
C:\Users\ASUS\Desktop\Project_Roadcrash\data\terrain\CAS_LINZ_terrain_enriched.csv

Final shape: (34866, 4)
